In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import cv2

In [ ]:
%%shell
pip install cython
pip install -U 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'

In [ ]:
%%shell

# Download TorchVision repo to use some files from
# references/detection
# Only clone if the directory doesn't exist
if [ ! -d "vision" ]; then
  git clone https://github.com/pytorch/vision.git
fi
cd vision
git checkout v0.3.0

cp references/detection/utils.py ../
cp references/detection/transforms.py ../
cp references/detection/coco_eval.py ../
cp references/detection/engine.py ../
cp references/detection/coco_utils.py ../

In [ ]:
# ================================================================
# Loading the model
# ================================================================

import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def get_instance_segmentation_model(num_classes):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
    return model

# 0 (background) + 5 tools = 6 classes
num_classes = 6
model = get_instance_segmentation_model(num_classes)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# Load your trained weights
model.load_state_dict(torch.load("/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/model_frames_1_5class_5july_new15_model.pt", map_location=device))

model.eval()

In [ ]:
# ================================================================
# Ground Truth Extractor Class
# ================================================================


import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
from scipy import ndimage
import re
import json

def natural_sort_key(s):
    """
    Splits a string into text and integers for natural sorting.
    Ensures 'img2.jpg' comes before 'img10.jpg' even without zero-padding.
    """
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', str(s))]

# ================================================================
# CONFIGURATION
# ================================================================
MASKS_ROOT = "/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/PerformanceScoreMasks"
IMAGES_ROOT = "/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/PerformanceScoreImages"

# Color-to-Label mapping (RGB values — verified from actual masks)
# Verified unique non-black colors across all masks:
#   (128, 0, 0)     = Red          → Suction
#   (0, 128, 0)     = Green        → Bipolar Forceps
#   (0, 128, 128)   = Cyan         → Scissor
#   (128, 128, 128) = Gray         → Dissecting Forceps
#   (64, 0, 0)      = Dark Maroon  → Needle Holder
COLOR_TO_LABEL = {
    (128, 0, 0):     1,   # Red → Suction
    (0, 128, 0):     2,   # Green → Bipolar Forceps
    (0, 128, 128):   3,   # Cyan → Scissor
    (128, 128, 128): 4,   # Gray → Dissecting Forceps
    (64, 0, 0):      5,   # Dark Maroon → Needle Holder
}

LABEL_TO_NAME = {
    1: 'Suction',
    2: 'Bipolar Forceps',
    3: 'Scissor',
    4: 'Dissecting Forceps',
    5: 'Needle Holder',
}

TOOL_COLORS_VIS = {
    1: 'red',        # Suction
    2: 'lime',       # Bipolar Forceps
    3: 'cyan',       # Scissor
    4: 'gray',       # Dissecting Forceps
    5: 'magenta',    # Needle Holder
}


class GroundTruthExtractor:
    """
    Extracts ground truth bounding boxes from color-coded mask images.

    Expects a root directory containing subdirectories (one per surgery/clip),
    each containing mask image files.

    Usage:
        extractor = GroundTruthExtractor(MASKS_ROOT, IMAGES_ROOT, COLOR_TO_LABEL, LABEL_TO_NAME)
        extractor.discover_colors()            # Step 1: Scan all masks for colors
        extractor.extract_all()                # Step 2: Extract GT boxes per surgery
        extractor.print_summary()              # Print per-surgery and overall stats
        extractor.visualize(num_per_surgery=2) # Step 3: Visual sanity check

        # Access results:
        extractor.ground_truth_per_surgery     # dict: {surgery_name: {filename: [boxes]}}
        extractor.get_combined()               # dict: {filename: [boxes]} (all surgeries merged)
    """

    def __init__(self, masks_root, images_root, color_to_label, label_to_name, tolerance=15, min_pixels=50):
        self.masks_root = masks_root
        self.images_root = images_root
        self.color_to_label = color_to_label
        self.label_to_name = label_to_name
        self.tolerance = tolerance
        self.min_pixels = min_pixels

        # Results storage
        self.ground_truth_per_surgery = {}   # {surgery_name: {filename: [boxes]}}
        self.surgery_dirs = []               # List of surgery subdirectory names
        self.all_colors = set()              # All discovered non-black colors

        # Discover surgery subdirectories
        self._discover_surgery_dirs()

    def _discover_surgery_dirs(self):
        """Find all subdirectories in the masks root."""
        self.surgery_dirs = sorted([
            d for d in os.listdir(self.masks_root)
            if os.path.isdir(os.path.join(self.masks_root, d))
        ], key=natural_sort_key)
        print(f"Found {len(self.surgery_dirs)} surgery subdirectories: {self.surgery_dirs}")

    def _get_image_files(self, surgery_name):
        """Get sorted list of image files in a surgery subdirectory."""
        mask_dir = os.path.join(self.masks_root, surgery_name)
        return sorted([
            f for f in os.listdir(mask_dir)
            if f.lower().endswith(('.jpg', '.png', '.jpeg'))
        ], key=natural_sort_key)

    # ==================================================================
    # STEP 1: Color Discovery
    # ==================================================================
    def discover_colors(self):
        """
        Scan ALL mask images across ALL surgery subdirectories
        to discover every unique non-black color.
        """
        print("\n--- Discovering unique colors across ALL masks ---")
        self.all_colors = set()
        total_files = 0

        for surgery_name in self.surgery_dirs:
            files = self._get_image_files(surgery_name)
            mask_dir = os.path.join(self.masks_root, surgery_name)

            for f in files:
                mask = Image.open(os.path.join(mask_dir, f)).convert('RGB')
                arr = np.array(mask)
                uniq = np.unique(arr.reshape(-1, 3), axis=0)
                for c in uniq:
                    t = tuple(c)
                    if t != (0, 0, 0):  # Skip background
                        self.all_colors.add(t)
                total_files += 1

        print(f"Scanned {total_files} mask files across {len(self.surgery_dirs)} surgeries")
        print(f"Unique non-black colors found: {sorted(self.all_colors)}")

        # Print mapping
        print("\nColor → Label mapping being used:")
        for color, label in self.color_to_label.items():
            name = self.label_to_name.get(label, f"Unknown-{label}")
            found_marker = "✓" if color in self.all_colors else "✗ (not found in masks)"
            print(f"  RGB{color} → {label} ({name}) {found_marker}")

        # Check for unmapped colors
        unmapped = self.all_colors - set(self.color_to_label.keys())
        if unmapped:
            print(f"\n⚠️  WARNING: The following colors are in your masks but NOT in COLOR_TO_LABEL:")
            print(f"  {unmapped}")
            print("  → You need to add these to COLOR_TO_LABEL and re-run!")
        else:
            print("\n✅ All colors are mapped. Proceeding.")

        return self.all_colors

    # ==================================================================
    # STEP 2: Bounding Box Extraction
    # ==================================================================
    def _extract_boxes_from_single_mask(self, mask_path):
        """
        Extract ground truth bounding boxes from a single color-coded mask image.

        Returns:
            list of dicts: [{'bbox': [x, y, w, h], 'label': int, 'mask_pixels': int}, ...]
        """
        mask = Image.open(mask_path).convert('RGB')
        mask_arr = np.array(mask)

        gt_boxes = []

        for color, label in self.color_to_label.items():
            # Create binary mask for this color with tolerance for JPEG artifacts
            color_arr = np.array(color)
            lower = np.clip(color_arr.astype(int) - self.tolerance, 0, 255)
            upper = np.clip(color_arr.astype(int) + self.tolerance, 0, 255)

            binary = np.all((mask_arr >= lower) & (mask_arr <= upper), axis=2).astype(np.uint8)

            if binary.sum() < self.min_pixels:
                continue

            # Connected components to find separate blobs
            labeled_array, num_features = ndimage.label(binary)

            for component_id in range(1, num_features + 1):
                component = (labeled_array == component_id)

                if component.sum() < self.min_pixels:
                    continue

                rows = np.where(component.any(axis=1))[0]
                cols = np.where(component.any(axis=0))[0]

                y_min, y_max = rows[0], rows[-1]
                x_min, x_max = cols[0], cols[-1]

                gt_boxes.append({
                    'bbox': [float(x_min), float(y_min), float(x_max - x_min), float(y_max - y_min)],
                    'label': label,
                    'mask_pixels': int(component.sum())
                })

        # Merge boxes of the same label (e.g., two forceps blades → one box)
        return self._merge_same_label_boxes(gt_boxes)

    def _merge_same_label_boxes(self, boxes):
        """Merge all boxes of the same label into a single encompassing bounding box."""
        if len(boxes) <= 1:
            return boxes

        labels = set(b['label'] for b in boxes)
        merged = []

        for label in labels:
            label_boxes = [b for b in boxes if b['label'] == label]

            if len(label_boxes) == 1:
                merged.append(label_boxes[0])
                continue

            # Merge ALL boxes of the same label into one encompassing box
            all_x_min = min(b['bbox'][0] for b in label_boxes)
            all_y_min = min(b['bbox'][1] for b in label_boxes)
            all_x_max = max(b['bbox'][0] + b['bbox'][2] for b in label_boxes)
            all_y_max = max(b['bbox'][1] + b['bbox'][3] for b in label_boxes)
            total_pixels = sum(b['mask_pixels'] for b in label_boxes)

            merged.append({
                'bbox': [all_x_min, all_y_min, all_x_max - all_x_min, all_y_max - all_y_min],
                'label': label,
                'mask_pixels': total_pixels
            })

        return merged

    def extract_all(self):
        """
        Extract GT boxes from ALL masks, organized by surgery subdirectory.
        Stores results in self.ground_truth_per_surgery.
        """
        print("\n--- Extracting ground truth boxes from all masks ---")
        self.ground_truth_per_surgery = {}

        for surgery_name in self.surgery_dirs:
            mask_dir = os.path.join(self.masks_root, surgery_name)
            files = self._get_image_files(surgery_name)

            surgery_gt = {}
            for fname in files:
                mask_path = os.path.join(mask_dir, fname)
                gt_boxes = self._extract_boxes_from_single_mask(mask_path)
                surgery_gt[fname] = gt_boxes

            self.ground_truth_per_surgery[surgery_name] = surgery_gt
            total_boxes = sum(len(v) for v in surgery_gt.values())
            print(f"  [{surgery_name}] {len(files)} frames → {total_boxes} GT boxes")

        print(f"\nDone. Extracted GT for {len(self.surgery_dirs)} surgeries.")
        return self.ground_truth_per_surgery

    # ==================================================================
    # Accessors
    # ==================================================================
    def get_combined(self):
        """
        Return a flat dict combining all surgeries: {filename: [boxes]}
        Filenames are prefixed with surgery name to avoid collisions.
        """
        combined = {}
        for surgery_name, surgery_gt in self.ground_truth_per_surgery.items():
            for fname, boxes in surgery_gt.items():
                # Prefix with surgery name to ensure uniqueness
                combined[f"{surgery_name}/{fname}"] = boxes
        return combined

    def get_surgery_gt(self, surgery_name):
        """Get ground truth for a specific surgery."""
        return self.ground_truth_per_surgery.get(surgery_name, {})

    # ==================================================================
    # STEP 3: Summary & Visualization
    # ==================================================================
    def print_summary(self):
        """Print per-surgery and overall statistics."""
        print("\n" + "=" * 60)
        print("GROUND TRUTH SUMMARY")
        print("=" * 60)

        overall_class_counts = defaultdict(int)
        overall_total = 0

        for surgery_name in self.surgery_dirs:
            surgery_gt = self.ground_truth_per_surgery.get(surgery_name, {})
            total_boxes = sum(len(v) for v in surgery_gt.values())
            overall_total += total_boxes

            class_counts = defaultdict(int)
            for fname, boxes in surgery_gt.items():
                for b in boxes:
                    class_counts[b['label']] += 1
                    overall_class_counts[b['label']] += 1

            print(f"\n  [{surgery_name}] — {len(surgery_gt)} frames, {total_boxes} boxes")
            for label, count in sorted(class_counts.items()):
                print(f"    {self.label_to_name.get(label, f'Label-{label}')}: {count}")

        print(f"\n  OVERALL — {overall_total} total GT boxes")
        for label, count in sorted(overall_class_counts.items()):
            print(f"    {self.label_to_name.get(label, f'Label-{label}')}: {count}")

    def visualize(self, num_per_surgery=2):
        """Show sample GT box visualizations from each surgery."""
        import random

        for surgery_name in self.surgery_dirs:
            surgery_gt = self.ground_truth_per_surgery.get(surgery_name, {})
            if not surgery_gt:
                continue

            # Pick random samples
            fnames_with_boxes = [f for f, b in surgery_gt.items() if len(b) > 0]
            if not fnames_with_boxes:
                continue

            samples = random.sample(fnames_with_boxes, min(num_per_surgery, len(fnames_with_boxes)))

            for fname in samples:
                gt_boxes = surgery_gt[fname]
                img_path = os.path.join(self.images_root, surgery_name, fname)
                mask_path = os.path.join(self.masks_root, surgery_name, fname)

                # Check that the image file exists
                if not os.path.exists(img_path):
                    print(f"  ⚠️ Image not found: {img_path}")
                    continue

                img = Image.open(img_path).convert('RGB')
                mask = Image.open(mask_path).convert('RGB')

                fig, axes = plt.subplots(1, 2, figsize=(20, 8))

                # Left: Image with GT boxes
                axes[0].imshow(np.array(img))
                axes[0].set_title(f"[{surgery_name}] {fname}", fontsize=12)
                for b in gt_boxes:
                    x, y, w, h = b['bbox']
                    label = b['label']
                    color = TOOL_COLORS_VIS.get(label, 'white')
                    name = self.label_to_name.get(label, f'L{label}')
                    rect = patches.Rectangle((x, y), w, h, linewidth=2,
                                             edgecolor=color, facecolor='none')
                    axes[0].add_patch(rect)
                    axes[0].text(x, y - 5, name, color='white', fontsize=10,
                                 bbox=dict(facecolor=color, alpha=0.7, pad=1))
                axes[0].axis('off')

                # Right: Mask
                axes[1].imshow(np.array(mask))
                axes[1].set_title(f"Mask: {fname}", fontsize=12)
                axes[1].axis('off')

                plt.tight_layout()
                plt.show()

    # ==================================================================
    # Save / Load Ground Truth
    # ==================================================================
    def save_to_json(self, output_dir, filename="ground_truth.json"):
        """Save extracted ground truth to a JSON file."""
        if not self.ground_truth_per_surgery:
            print("⚠️ No ground truth to save. Run extract_all() first.")
            return
        os.makedirs(output_dir, exist_ok=True)
        path = os.path.join(output_dir, filename)
        with open(path, 'w') as f:
            json.dump(self.ground_truth_per_surgery, f, indent=2)
        print(f"\n📁 Ground truth saved to: {path}")

    def load_from_json(self, json_path):
        """Load ground truth from a JSON file, bypassing extraction."""
        if not os.path.exists(json_path):
            print(f"⚠️ File not found: {json_path}")
            return False
        with open(json_path, 'r') as f:
            self.ground_truth_per_surgery = json.load(f)
        print(f"\n📁 Loaded ground truth for {len(self.ground_truth_per_surgery)} surgeries from {json_path}")
        return True

In [ ]:
# ================================================================
# Create the Ground Truth Extractor
# ================================================================


gt_extractor = GroundTruthExtractor(
    masks_root=MASKS_ROOT,
    images_root=IMAGES_ROOT,
    color_to_label=COLOR_TO_LABEL,
    label_to_name=LABEL_TO_NAME
)

# Step 1: Discover colors across ALL masks
# gt_extractor.discover_colors()

# Step 2: Extract GT boxes (per surgery)
# gt_extractor.extract_all()


# Step 3: Save to JSON (so you don't have to re-extract next time)
# gt_extractor.save_to_json("/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources")

# ---------------------------------------------------------
# OPTION B: Load from saved JSON (skip Steps 1-3 above)
# ---------------------------------------------------------

gt_extractor.load_from_json("/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/ground_truth.json")



# Print summary
gt_extractor.print_summary()

# Access results:
# gt_extractor.ground_truth_per_surgery  → {surgery_name: {filename: [boxes]}}
# gt_extractor.get_surgery_gt("surgery_1") → {filename: [boxes]} for one surgery
# gt_extractor.get_combined()             → {surgery/filename: [boxes]} flat dict


# Visualization just for checking

# Show 2 random samples from each surgery
gt_extractor.visualize(num_per_surgery=2)

In [ ]:
# ================================================================
# Evaluator Class
# ================================================================

import json
import os
import numpy as np
from collections import defaultdict
from datetime import datetime


class DetectionEvaluator:
    """
    Evaluates object detection predictions against ground truth.

    Computes: IoU, Precision, Recall, F1, mAP@0.5, mAP@0.75,
    mAP@[.5:.95], Classification Accuracy, Mean IoU, Confusion Matrix.

    Stores all results internally and can save them to JSON files.
    """

    def __init__(self, label_to_name, output_dir=None):
        """
        Args:
            label_to_name: dict mapping label IDs to tool names, e.g. {1: 'Suction', ...}
            output_dir: path to save JSON result files (optional, created if doesn't exist)
        """
        self.label_to_name = label_to_name
        self.output_dir = output_dir
        self.all_results = {}  # {method_name: results_dict}

        if output_dir:
            os.makedirs(output_dir, exist_ok=True)

    # ==================================================================
    # CORE: IoU Computation
    # ==================================================================
    @staticmethod
    def compute_iou(box1, box2):
        """
        Computes IoU between two boxes in [x, y, w, h] format.
        """
        x1, y1, w1, h1 = box1
        x2, y2, w2, h2 = box2

        # Convert to [x_min, y_min, x_max, y_max]
        x1_max, y1_max = x1 + w1, y1 + h1
        x2_max, y2_max = x2 + w2, y2 + h2

        # Intersection
        xA = max(x1, x2)
        yA = max(y1, y2)
        xB = min(x1_max, x2_max)
        yB = min(y1_max, y2_max)

        inter_w = max(0, xB - xA)
        inter_h = max(0, yB - yA)
        inter_area = inter_w * inter_h

        # Union
        area1 = w1 * h1
        area2 = w2 * h2
        union_area = area1 + area2 - inter_area

        if union_area <= 0:
            return 0.0

        return inter_area / union_area

    # ==================================================================
    # CORE: Greedy Matching
    # ==================================================================
    @staticmethod
    def match_predictions_to_gt(preds, gts, iou_threshold=0.5):
        """
        Matches predictions to ground truth using greedy IoU matching.
        Predictions are sorted by score (highest first).

        Returns:
            tp_list: list of dicts with 'pred', 'gt', 'iou', 'label_correct'
            fp_list: list of unmatched predictions
            fn_list: list of unmatched ground truths
        """
        if len(preds) == 0:
            return [], [], list(gts)

        if len(gts) == 0:
            return [], list(preds), []

        # Sort predictions by score (highest first) for greedy matching
        sorted_preds = sorted(preds, key=lambda p: p.get('score', 1.0), reverse=True)

        gt_matched = [False] * len(gts)
        tp_list = []
        fp_list = []

        for pred in sorted_preds:
            best_iou = 0.0
            best_gt_idx = -1

            for gt_idx, gt in enumerate(gts):
                if gt_matched[gt_idx]:
                    continue

                iou = DetectionEvaluator.compute_iou(pred['bbox'], gt['bbox'])

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx

            if best_iou >= iou_threshold and best_gt_idx != -1:
                gt_matched[best_gt_idx] = True
                tp_list.append({
                    'pred': pred,
                    'gt': gts[best_gt_idx],
                    'iou': best_iou,
                    'label_correct': pred['label'] == gts[best_gt_idx]['label']
                })
            else:
                fp_list.append(pred)

        # Unmatched ground truths = false negatives
        fn_list = [gts[i] for i in range(len(gts)) if not gt_matched[i]]

        return tp_list, fp_list, fn_list

    # ==================================================================
    # CORE: Evaluate at a single IoU threshold
    # ==================================================================
    def _evaluate_at_threshold(self, predictions, ground_truth, iou_threshold=0.5):
        """
        Full evaluation of predictions against ground truth at one IoU threshold.

        Args:
            predictions: dict {filename: [{'bbox': [x,y,w,h], 'label': int, 'score': float}, ...]}
            ground_truth: dict {filename: [{'bbox': [x,y,w,h], 'label': int}, ...]}

        Returns:
            dict with all computed metrics at this threshold
        """
        all_tp = []
        all_fp = []
        all_fn = []

        per_class_tp = defaultdict(list)
        per_class_fp = defaultdict(list)
        per_class_fn = defaultdict(list)

        # Confusion matrix: confusion[gt_label][pred_label] = count
        confusion = defaultdict(lambda: defaultdict(int))
        iou_values = []

        for fname in ground_truth:
            gts = ground_truth[fname]
            preds = predictions.get(fname, [])

            tp, fp, fn = self.match_predictions_to_gt(preds, gts, iou_threshold)

            all_tp.extend(tp)
            all_fp.extend(fp)
            all_fn.extend(fn)

            # Per-class aggregation
            for match in tp:
                gt_label = match['gt']['label']
                pred_label = match['pred']['label']
                per_class_tp[gt_label].append(match)
                confusion[gt_label][pred_label] += 1
                iou_values.append(match['iou'])

            for pred in fp:
                per_class_fp[pred['label']].append(pred)
                confusion[0][pred['label']] += 1  # 0 = no GT (false positive)

            for gt in fn:
                per_class_fn[gt['label']].append(gt)
                confusion[gt['label']][0] += 1  # 0 = no prediction (miss)

        # ---- Aggregate Metrics ----
        total_tp = len(all_tp)
        total_fp = len(all_fp)
        total_fn = len(all_fn)

        precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
        recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        # Classification accuracy (among matched detections)
        correct_labels = sum(1 for m in all_tp if m['label_correct'])
        classification_acc = correct_labels / total_tp if total_tp > 0 else 0.0

        mean_iou = float(np.mean(iou_values)) if iou_values else 0.0

        # ---- Per-Class Metrics ----
        per_class_metrics = {}
        all_labels = set(list(per_class_tp.keys()) + list(per_class_fn.keys()))

        for label in sorted(all_labels):
            tp_count = len(per_class_tp[label])
            fp_count = len(per_class_fp[label])
            fn_count = len(per_class_fn[label])

            p = tp_count / (tp_count + fp_count) if (tp_count + fp_count) > 0 else 0.0
            r = tp_count / (tp_count + fn_count) if (tp_count + fn_count) > 0 else 0.0
            f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

            class_ious = [m['iou'] for m in per_class_tp[label]]
            class_mean_iou = float(np.mean(class_ious)) if class_ious else 0.0

            correct_in_class = sum(1 for m in per_class_tp[label] if m['label_correct'])
            class_acc = correct_in_class / tp_count if tp_count > 0 else 0.0

            per_class_metrics[str(label)] = {
                'name': self.label_to_name.get(label, f'Label-{label}'),
                'tp': tp_count,
                'fp': fp_count,
                'fn': fn_count,
                'precision': round(p, 4),
                'recall': round(r, 4),
                'f1': round(f, 4),
                'mean_iou': round(class_mean_iou, 4),
                'classification_accuracy': round(class_acc, 4)
            }

        # Convert confusion to JSON-serializable format
        confusion_serializable = {}
        for gt_l, pred_dict in confusion.items():
            confusion_serializable[str(gt_l)] = {str(p_l): c for p_l, c in pred_dict.items()}

        return {
            'iou_threshold': iou_threshold,
            'total_tp': total_tp,
            'total_fp': total_fp,
            'total_fn': total_fn,
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'f1': round(f1, 4),
            'classification_accuracy': round(classification_acc, 4),
            'mean_iou': round(mean_iou, 4),
            'per_class': per_class_metrics,
            'confusion': confusion_serializable
        }

    # ==================================================================
    # PUBLIC: Evaluate a method (multi-threshold)
    # ==================================================================
    def evaluate(self, method_name, predictions, ground_truth):
        """
        Full evaluation of a method at IoU=0.5, 0.75, and [0.5:0.95].
        Stores results internally and optionally saves to JSON.

        Args:
            method_name: string identifier for this evaluation (e.g., "Raw Model")
            predictions: dict {filename: [{'bbox': [x,y,w,h], 'label': int, 'score': float}]}
            ground_truth: dict {filename: [{'bbox': [x,y,w,h], 'label': int}]}

        Returns:
            dict with all results
        """
        print(f"\n{'=' * 60}")
        print(f"EVALUATING: {method_name}")
        print(f"{'=' * 60}")

        # mAP@0.5
        result_50 = self._evaluate_at_threshold(predictions, ground_truth, iou_threshold=0.5)

        # mAP@0.75
        result_75 = self._evaluate_at_threshold(predictions, ground_truth, iou_threshold=0.75)

        # mAP@[0.5:0.95] (COCO-style)
        thresholds = np.arange(0.5, 1.0, 0.05).tolist()
        results_all = []
        for t in thresholds:
            r = self._evaluate_at_threshold(predictions, ground_truth, iou_threshold=t)
            results_all.append(r)

        avg_precision = round(float(np.mean([r['precision'] for r in results_all])), 4)
        avg_recall = round(float(np.mean([r['recall'] for r in results_all])), 4)
        avg_f1 = round(float(np.mean([r['f1'] for r in results_all])), 4)

        results = {
            'method_name': method_name,
            'timestamp': datetime.now().isoformat(),
            'num_images': len(ground_truth),
            'num_predictions': sum(len(v) for v in predictions.values()),
            'num_gt_boxes': sum(len(v) for v in ground_truth.values()),
            'mAP@0.5': result_50,
            'mAP@0.75': result_75,
            'mAP@[.5:.95]': {
                'precision': avg_precision,
                'recall': avg_recall,
                'f1': avg_f1
            }
        }

        # Store internally
        self.all_results[method_name] = results

        # Print summary
        self._print_single_result(results)

        # Save to JSON
        if self.output_dir:
            safe_name = method_name.replace(' ', '_').replace('/', '_').replace('\\', '_')
            json_path = os.path.join(self.output_dir, f"{safe_name}.json")
            with open(json_path, 'w') as f:
                json.dump(results, f, indent=2)
            print(f"\n📁 Results saved to: {json_path}")

        return results

    # ==================================================================
    # PRINTING: Single result
    # ==================================================================
    def _print_single_result(self, results):
        """Print formatted results for a single method."""
        r50 = results['mAP@0.5']
        r75 = results['mAP@0.75']
        coco = results['mAP@[.5:.95]']

        print(f"\n--- Overall Metrics ---")
        print(f"{'Metric':<30} {'@IoU=0.5':>10} {'@IoU=0.75':>10}")
        print("-" * 52)
        print(f"{'Precision':<30} {r50['precision']:>10.4f} {r75['precision']:>10.4f}")
        print(f"{'Recall':<30} {r50['recall']:>10.4f} {r75['recall']:>10.4f}")
        print(f"{'F1-Score':<30} {r50['f1']:>10.4f} {r75['f1']:>10.4f}")
        print(f"{'Classification Accuracy':<30} {r50['classification_accuracy']:>10.4f} {r75['classification_accuracy']:>10.4f}")
        print(f"{'Mean IoU (matched)':<30} {r50['mean_iou']:>10.4f} {r75['mean_iou']:>10.4f}")
        print(f"{'TP / FP / FN':<30} {r50['total_tp']:>3}/{r50['total_fp']:>3}/{r50['total_fn']:>3}    {r75['total_tp']:>3}/{r75['total_fp']:>3}/{r75['total_fn']:>3}")

        print(f"\n--- Per-Class Metrics @IoU=0.5 ---")
        print(f"{'Tool':<25} {'Prec':>7} {'Recall':>7} {'F1':>7} {'mIoU':>7} {'TP':>4} {'FP':>4} {'FN':>4}")
        print("-" * 70)
        for label_key in sorted(r50['per_class'].keys(), key=lambda x: int(x)):
            m = r50['per_class'][label_key]
            print(f"{m['name']:<25} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f1']:>7.4f} {m['mean_iou']:>7.4f} {m['tp']:>4} {m['fp']:>4} {m['fn']:>4}")

        print(f"\n--- COCO-style mAP@[.5:.95] ---")
        print(f"  Average Precision: {coco['precision']:.4f}")
        print(f"  Average Recall:    {coco['recall']:.4f}")
        print(f"  Average F1:        {coco['f1']:.4f}")

    # ==================================================================
    # PRINTING: Comparison table across methods
    # ==================================================================
    def print_comparison(self):
        """Print a comparison table across all evaluated methods."""
        if not self.all_results:
            print("No results to compare. Run evaluate() first.")
            return

        print("\n" + "=" * 90)
        print("COMPARISON TABLE")
        print("=" * 90)
        print(f"\n{'Method':<35} {'Prec@0.5':>9} {'Rec@0.5':>9} {'F1@0.5':>9} {'mIoU':>9} {'Cls Acc':>9}")
        print("-" * 82)

        for method_name, result in self.all_results.items():
            r = result['mAP@0.5']
            print(f"{method_name:<35} {r['precision']:>9.4f} {r['recall']:>9.4f} {r['f1']:>9.4f} {r['mean_iou']:>9.4f} {r['classification_accuracy']:>9.4f}")

    # ==================================================================
    # EXPORT: LaTeX table for paper
    # ==================================================================
    def export_latex(self):
        """Print LaTeX table code ready to paste into your paper."""
        if not self.all_results:
            print("No results to export. Run evaluate() first.")
            return

        print("\n% ====== LaTeX Table (copy into your paper) ======")
        print("\\begin{table}[htbp]")
        print("\\centering")
        print("\\caption{Detection performance comparison of post-processing methods}")
        print("\\label{tab:detection_results}")
        print("\\begin{tabular}{lccccc}")
        print("\\hline")
        print("\\textbf{Method} & \\textbf{Prec} & \\textbf{Recall} & \\textbf{F1} & \\textbf{mIoU} & \\textbf{Cls Acc} \\\\")
        print("\\hline")

        for method_name, result in self.all_results.items():
            r = result['mAP@0.5']
            safe_name = method_name.replace('_', '\\_')
            print(f"{safe_name} & {r['precision']:.3f} & {r['recall']:.3f} & {r['f1']:.3f} & {r['mean_iou']:.3f} & {r['classification_accuracy']:.3f} \\\\")

        print("\\hline")
        print("\\end{tabular}")
        print("\\end{table}")

    # ==================================================================
    # EXPORT: Per-class LaTeX table for paper
    # ==================================================================
    def export_per_class_latex(self, method_name=None):
        """Print a per-class LaTeX table for a specific method (or the first one)."""
        if method_name is None:
            method_name = list(self.all_results.keys())[0]

        result = self.all_results.get(method_name)
        if not result:
            print(f"No results for '{method_name}'")
            return

        r50 = result['mAP@0.5']

        print(f"\n% ====== Per-Class LaTeX Table for: {method_name} ======")
        print("\\begin{table}[htbp]")
        print("\\centering")
        print(f"\\caption{{Per-class detection results — {method_name}}}")
        print(f"\\label{{tab:per_class_{method_name.replace(' ', '_').lower()}}}")
        print("\\begin{tabular}{lcccccc}")
        print("\\hline")
        print("\\textbf{Tool} & \\textbf{Prec} & \\textbf{Recall} & \\textbf{F1} & \\textbf{mIoU} & \\textbf{TP} & \\textbf{FN} \\\\")
        print("\\hline")

        for label_key in sorted(r50['per_class'].keys(), key=lambda x: int(x)):
            m = r50['per_class'][label_key]
            safe_name = m['name'].replace('_', '\\_')
            print(f"{safe_name} & {m['precision']:.3f} & {m['recall']:.3f} & {m['f1']:.3f} & {m['mean_iou']:.3f} & {m['tp']} & {m['fn']} \\\\")

        print("\\hline")
        print("\\end{tabular}")
        print("\\end{table}")

    # ==================================================================
    # SAVE: Export all results to a combined JSON
    # ==================================================================
    def save_all_results(self, filename="all_results.json"):
        """Save all method results to a single combined JSON file."""
        if self.output_dir:
            path = os.path.join(self.output_dir, filename)
        else:
            path = filename

        with open(path, 'w') as f:
            json.dump(self.all_results, f, indent=2)
        print(f"📁 All results saved to: {path}")

In [ ]:
# ================================================================
# Create Evaluator
# ================================================================

RESULTS_DIR = "/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/evaluation_results"

evaluator = DetectionEvaluator(
    label_to_name=LABEL_TO_NAME,
    output_dir=RESULTS_DIR
)

In [ ]:
# ================================================================
# Get any image prediction
# ================================================================

import os
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

# 1. Pick ANY image path directly
test_image_path = "/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/PerformanceScoreImages/surgery_1/2_inst_16_44.jpg" # Change this to any image!

# 2. Load and transform the image
img_pil = Image.open(test_image_path).convert("RGB")
transform = T.Compose([T.ToTensor()])
img_tensor = transform(img_pil)

# 3. Run inference
model.eval()
with torch.no_grad():
    prediction = model([img_tensor.to(device)])

# 4. Print results
print("Predicted Labels:", prediction[0]['labels'].cpu().numpy())
print("Predicted Scores:", [f"{s:.4f}" for s in prediction[0]['scores'].cpu().numpy()])

# 5. Visualize Image and First Mask
masks = prediction[0]['masks'].cpu().detach().numpy()

plt.figure(figsize=(12, 6))

# Left: Original Image
plt.subplot(1, 2, 1)
plt.imshow(img_pil)
plt.title("Original Image")
plt.axis('off')

# Right: First Predicted Mask
plt.subplot(1, 2, 2)
if len(masks) > 0:
    plt.imshow(masks[2, 0, :, :])
    plt.title(f"First Predicted Mask (Label: {prediction[0]['labels'][0].item()})")
else:
    plt.title("No tools detected")
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Detection Visualizer Class
# ================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import os
from PIL import Image


class DetectionVisualizer:
    """
    Presentation-quality visualization of detection predictions.

    Usage:
        viz = DetectionVisualizer(LABEL_TO_NAME, IMAGES_ROOT)

        # Single frame:
        viz.show_frame("surgery_1", preds_video, frame_id=5)

        # With ground truth overlay:
        viz.show_frame("surgery_1", preds_video, frame_id=5, gt=gt_dict)

        # Range of frames:
        viz.show_range("surgery_1", preds_video, start=1, end=10)

        # Side-by-side comparison (before vs after post-processing):
        viz.compare("surgery_1", baseline_pv, processed_pv, frame_id=5,
                     titles=["Raw Model", "DSU + Gap Fill"])

        # Grid view:
        viz.show_grid("surgery_1", preds_video, frame_ids=[1, 5, 10, 15])
    """

    # Professional color palette for each tool
    TOOL_COLORS = {
        1: '#E74C3C',   # Suction — Red
        2: '#2ECC71',   # Bipolar Forceps — Green
        3: '#00BCD4',   # Scissor — Cyan
        4: '#95A5A6',   # Dissecting Forceps — Gray
        5: '#8E44AD',   # Needle Holder — Purple
    }

    def __init__(self, label_to_name, images_root):
        """
        Args:
            label_to_name: dict mapping label IDs to tool names
            images_root: root directory containing surgery subdirectories
        """
        self.label_to_name = label_to_name
        self.images_root = images_root

    def _get_sorted_files(self, surgery_name):
        """Get sorted image filenames for a surgery."""
        img_dir = os.path.join(self.images_root, surgery_name)
        files = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith(('.jpg', '.png', '.jpeg'))
        ], key=natural_sort_key)
        return files

    def _load_image(self, surgery_name, frame_id):
        """Load image by frame_id (1-indexed)."""
        files = self._get_sorted_files(surgery_name)
        if frame_id < 1 or frame_id > len(files):
            print(f"⚠️ Frame {frame_id} out of range (1-{len(files)})")
            return None, None
        fname = files[frame_id - 1]
        img_path = os.path.join(self.images_root, surgery_name, fname)
        img = np.array(Image.open(img_path).convert("RGB"))
        return img, fname

    def _draw_predictions(self, ax, preds_video, frame_id, alpha=1.0,
                          linestyle='-', linewidth=2.5):
        """Draw prediction boxes on an axis."""
        if frame_id not in preds_video:
            return

        detections = preds_video[frame_id]
        for det in detections:
            if 'scores' not in det:
                continue
            label_id = int(np.argmax(det['scores']))
            if label_id == 0:
                continue  # Skip background

            tool_name = self.label_to_name.get(label_id, f"Unknown-{label_id}")
            color = self.TOOL_COLORS.get(label_id, '#FFFFFF')

            x, y, w, h = det['bbox']

            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=linewidth,
                edgecolor=color,
                facecolor='none',
                linestyle=linestyle,
                alpha=alpha
            )
            ax.add_patch(rect)

            ax.text(
                x, y - 6,
                tool_name,
                color='white',
                fontsize=9,
                fontweight='bold',
                bbox=dict(
                    facecolor=color, edgecolor='none',
                    alpha=0.85 * alpha, pad=2,
                    boxstyle='round,pad=0.3'
                )
            )

    def _draw_gt(self, ax, gt_dict, fname):
        """Draw ground truth boxes on an axis (dashed yellow)."""
        if fname not in gt_dict:
            return

        for box in gt_dict[fname]:
            x, y, w, h = box['bbox']
            label_id = box['label']
            tool_name = self.label_to_name.get(label_id, f"?-{label_id}")

            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=2,
                edgecolor='yellow',
                facecolor='none',
                linestyle='--',
                alpha=0.9
            )
            ax.add_patch(rect)

            ax.text(
                x + w, y - 6,
                f"GT: {tool_name}",
                color='black',
                fontsize=8,
                fontweight='bold',
                ha='right',
                bbox=dict(
                    facecolor='yellow', edgecolor='none',
                    alpha=0.75, pad=2,
                    boxstyle='round,pad=0.3'
                )
            )

    # ==================================================================
    # PUBLIC: Show a single frame
    # ==================================================================
    def show_frame(self, surgery_name, preds_video, frame_id, gt=None,
                   figsize=(14, 10), title=None):
        """
        Visualize a single frame with predictions (and optionally GT).

        Args:
            surgery_name: name of the surgery subdirectory
            preds_video: {frame_id: [detections]} dict
            frame_id: 1-indexed frame number
            gt: optional ground truth dict {filename: [boxes]} for this surgery
            figsize: figure size
            title: custom title (auto-generated if None)
        """
        img, fname = self._load_image(surgery_name, frame_id)
        if img is None:
            return

        fig, ax = plt.subplots(1, figsize=figsize)
        ax.imshow(img)
        ax.axis('off')

        # Draw predictions
        self._draw_predictions(ax, preds_video, frame_id)

        # Draw ground truth if provided
        if gt is not None:
            self._draw_gt(ax, gt, fname)

        if title is None:
            n_det = len(preds_video.get(frame_id, []))
            title = f"{surgery_name} — Frame {frame_id} ({fname}) — {n_det} detection(s)"
            if gt is not None:
                n_gt = len(gt.get(fname, []))
                title += f" | {n_gt} GT"

        ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

        # Add legend
        legend_elements = []
        for label_id in sorted(self.TOOL_COLORS.keys()):
            name = self.label_to_name.get(label_id, '?')
            color = self.TOOL_COLORS[label_id]
            legend_elements.append(
                patches.Patch(facecolor=color, edgecolor=color, label=name)
            )
        if gt is not None:
            legend_elements.append(
                patches.Patch(facecolor='yellow', edgecolor='yellow',
                              label='Ground Truth', alpha=0.75)
            )
        ax.legend(handles=legend_elements, loc='lower right',
                  fontsize=9, framealpha=0.8)

        plt.tight_layout()
        plt.show()

    # ==================================================================
    # PUBLIC: Show a range of frames
    # ==================================================================
    def show_range(self, surgery_name, preds_video, start=1, end=None, gt=None):
        """Show all frames from start to end (inclusive)."""
        files = self._get_sorted_files(surgery_name)
        if end is None:
            end = len(files)
        for fid in range(start, end + 1):
            self.show_frame(surgery_name, preds_video, fid, gt=gt)

    # ==================================================================
    # PUBLIC: Side-by-side comparison
    # ==================================================================
    def compare(self, surgery_name, preds_video_1, preds_video_2, frame_id,
                titles=None, gt=None, figsize=(24, 10)):
        """
        Show two preds_video side by side for the same frame.
        Perfect for comparing Raw Model vs Post-Processed.
        """
        img, fname = self._load_image(surgery_name, frame_id)
        if img is None:
            return

        if titles is None:
            titles = ["Before", "After"]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

        for ax, pv, t in [(ax1, preds_video_1, titles[0]),
                           (ax2, preds_video_2, titles[1])]:
            ax.imshow(img)
            ax.axis('off')
            self._draw_predictions(ax, pv, frame_id)
            if gt is not None:
                self._draw_gt(ax, gt, fname)

            n_det = len(pv.get(frame_id, []))
            ax.set_title(f"{t} — Frame {frame_id} ({n_det} det.)",
                         fontsize=13, fontweight='bold', pad=10)

        plt.suptitle(f"{surgery_name} — Frame {frame_id}",
                     fontsize=15, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()

    # ==================================================================
    # PUBLIC: Grid view of selected frames
    # ==================================================================
    def show_grid(self, surgery_name, preds_video, frame_ids,
                  cols=4, figsize_per_cell=(5, 4), gt=None):
        """
        Show multiple frames in a compact grid layout.
        Great for paper figures showing temporal progression.
        """
        n = len(frame_ids)
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols,
                                 figsize=(figsize_per_cell[0] * cols,
                                          figsize_per_cell[1] * rows))
        if rows == 1 and cols == 1:
            axes = np.array([[axes]])
        elif rows == 1:
            axes = axes[np.newaxis, :]
        elif cols == 1:
            axes = axes[:, np.newaxis]

        for idx, fid in enumerate(frame_ids):
            r, c = idx // cols, idx % cols
            ax = axes[r, c]

            img, fname = self._load_image(surgery_name, fid)
            if img is not None:
                ax.imshow(img)
                self._draw_predictions(ax, preds_video, fid, linewidth=1.5)
                if gt is not None:
                    self._draw_gt(ax, gt, fname)
                n_det = len(preds_video.get(fid, []))
                ax.set_title(f"Frame {fid} ({n_det} det.)", fontsize=10)
            ax.axis('off')

        # Hide unused cells
        for idx in range(n, rows * cols):
            r, c = idx // cols, idx % cols
            axes[r, c].axis('off')

        plt.suptitle(f"{surgery_name} — Detection Results",
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()


In [ ]:
# ================================================================
# Create the visualizer (run once, use anywhere)
# ================================================================

viz = DetectionVisualizer(
    label_to_name=LABEL_TO_NAME,
    images_root=IMAGES_ROOT
)

In [ ]:
# ================================================================
# Baseline Predictor Class
# ================================================================

import torch
import pickle
import os
import numpy as np
from PIL import Image
import torchvision.transforms as T
import torch.nn.functional as F
from skimage.morphology import medial_axis, skeletonize


class BaselinePredictor:
    """
    Runs model inference on evaluation images, applies NMS, and produces
    predictions in both evaluator and preds_video formats.

    Usage:
        predictor = BaselinePredictor(model, device, num_classes=6, conf_threshold=0.3)
        predictor.run_inference(gt_extractor.surgery_dirs, IMAGES_ROOT)

        # For evaluation:
        raw_predictions = predictor.get_combined_eval_predictions()

        # For post-processing (per surgery):
        preds_video = predictor.get_preds_video("surgery_1")

        # Save to disk:
        predictor.save_all(output_dir)
    """

    def __init__(self, model, device, num_classes=6, conf_threshold=0.3,
                 mask_iou_threshold=0.2):
        """
        Args:
            model: trained Mask R-CNN model
            device: torch device
            num_classes: number of classes (including background as 0)
            conf_threshold: minimum confidence score to keep a prediction
            mask_iou_threshold: IoU threshold for mask-based NMS
        """
        self.model = model
        self.device = device
        self.num_classes = num_classes
        self.conf_threshold = conf_threshold
        self.mask_iou_threshold = mask_iou_threshold
        self.transform = T.Compose([T.ToTensor()])

        # Per-surgery results
        self.eval_predictions_per_surgery = {}   # {surgery: {filename: [eval_format]}}
        self.preds_video_per_surgery = {}         # {surgery: {frame_id: [preds_video_format]}}
        self.position_per_surgery = {}            # {surgery: np.array shape (num_classes, num_frames+1, 2)}

    # ==================================================================
    # Helper: Filter low-confidence predictions
    # ==================================================================
    @staticmethod
    def good_predictions(predictions, threshold):
        """Filter predictions below the confidence threshold."""
        new_prediction = predictions[0]
        scores = new_prediction['scores'].cpu().detach().numpy()
        keep_count = int(np.sum(scores >= threshold))

        new_prediction['labels'] = new_prediction['labels'][:keep_count]
        new_prediction['boxes'] = new_prediction['boxes'][:keep_count]
        new_prediction['masks'] = new_prediction['masks'][:keep_count]
        new_prediction['scores'] = new_prediction['scores'][:keep_count]
        return [new_prediction]

    # ==================================================================
    # Helper: Check if one box is inside another
    # ==================================================================
    @staticmethod
    def check_inside(box1, box2):
        """
        Check box containment. Box format: [xmin, ymin, xmax, ymax].
        Returns: 1 if box1 inside box2, 2 if box2 inside box1, 0 otherwise.
        """
        if (box2[0] >= box1[0] and box2[1] >= box1[1] and
            box2[2] <= box1[2] and box2[3] <= box1[3]):
            return 2  # Box 2 is inside Box 1

        elif (box1[0] >= box2[0] and box1[1] >= box2[1] and
              box1[2] <= box2[2] and box1[3] <= box2[3]):
            return 1  # Box 1 is inside Box 2

        return 0  # No containment

    # ==================================================================
    # Helper: Compute IoU between two binary masks
    # ==================================================================
    @staticmethod
    def iou_masks(mask1, mask2):
        """Compute IoU between two binary masks."""
        intersection = np.logical_and(mask1, mask2)
        union = np.logical_or(mask1, mask2)
        union_sum = np.sum(union)
        if union_sum == 0:
            return 0.0
        return np.sum(intersection) / union_sum

    # ==================================================================
    # Helper: Convert model output to preds_video format
    # ==================================================================
    def _to_preds_video_format(self, preds, ih, iw, frame_id):
        """
        Convert filtered model predictions to preds_video format
        (compatible with DSU and gap-filling methods).
        """
        preds_frame = []
        for i in range(len(preds[0]['boxes'])):
            x_min, y_min, x_max, y_max = preds[0]['boxes'][i].cpu().detach().numpy()
            width = x_max - x_min
            height = y_max - y_min

            if width <= 0 or height <= 0:
                continue

            width_diff = max(0, (ih - iw) // 2)
            height_diff = max(0, (iw - ih) // 2)

            bbox_center = [
                (x_min + width_diff + width / 2) / max(iw, ih),
                (y_min + height_diff + height / 2) / max(iw, ih)
            ]

            pred = {
                'image_id': frame_id,
                'bbox': [float(x_min), float(y_min), float(width), float(height)],
                'bbox_center': bbox_center,
                # One-hot encoding for DSU majority voting compatibility
                'scores': F.one_hot(
                    preds[0]['labels'][i], num_classes=self.num_classes
                ).cpu().detach().numpy()
            }
            preds_frame.append(pred)

        return preds_frame

    # ==================================================================
    # Helper: Convert model output to evaluator format
    # ==================================================================
    @staticmethod
    def _to_eval_format(preds):
        """
        Convert filtered model predictions to evaluator format:
        [{'bbox': [x, y, w, h], 'label': int, 'score': float}]
        """
        pred_boxes = []
        for i in range(len(preds[0]['labels'])):
            x1, y1, x2, y2 = preds[0]['boxes'][i].cpu().numpy()
            label = preds[0]['labels'][i].item()
            score = preds[0]['scores'][i].item()
            pred_boxes.append({
                'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                'label': label,
                'score': score
            })
        return pred_boxes

    # ==================================================================
    # Core: Apply Mask IoU NMS + Box Containment NMS
    # ==================================================================
    def _apply_nms(self, preds):
        """Apply Mask IoU NMS and Box Containment NMS to filtered predictions."""

        # --- Mask IoU NMS ---
        num_preds = len(preds[0]['labels'])
        keep_iou = torch.ones(num_preds, dtype=torch.bool)
        masks = preds[0]['masks'].cpu().detach().numpy()
        scores = preds[0]['scores']

        for i in range(0, num_preds - 1):
            if not keep_iou[i]:
                continue
            for j in range(i + 1, num_preds):
                if not keep_iou[j]:
                    continue
                iou_score = self.iou_masks(masks[i, 0, :, :], masks[j, 0, :, :])
                if iou_score > self.mask_iou_threshold:
                    if scores[i] > scores[j]:
                        keep_iou[j] = False
                    else:
                        keep_iou[i] = False
                        break

        for key in ['scores', 'labels', 'boxes', 'masks']:
            preds[0][key] = preds[0][key][keep_iou]

        # --- Box Containment NMS ---
        num_preds = len(preds[0]['labels'])
        keep_inside = torch.ones(num_preds, dtype=torch.bool)
        boxes = preds[0]['boxes'].cpu().detach().numpy()

        for i in range(0, num_preds - 1):
            if not keep_inside[i]:
                continue
            for j in range(i + 1, num_preds):
                if not keep_inside[j]:
                    continue
                inside_index = self.check_inside(boxes[i, :], boxes[j, :])
                if inside_index == 1:
                    keep_inside[i] = False
                    break
                elif inside_index == 2:
                    keep_inside[j] = False

        for key in ['scores', 'labels', 'boxes', 'masks']:
            preds[0][key] = preds[0][key][keep_inside]

        return preds

    # ==================================================================
    # Helper: Compute skeleton endpoints for tool tip tracking
    # ==================================================================
    @staticmethod
    def _compute_skeleton_position(preds, frame_index, position):
        """
        For each detected tool, skeletonize the predicted mask,
        find endpoints, pick the closest to image center, and
        store in the position array.

        Args:
            preds: filtered model predictions (with masks)
            frame_index: 0-based frame index (for position array indexing)
            position: np.array of shape (num_classes, num_frames+1, 2)
        """
        masks = preds[0]['masks'].cpu().detach().numpy()
        labels = preds[0]['labels'].cpu().detach().numpy()

        for i in range(len(labels)):
            mask1 = masks[i, 0, :, :]
            mask1[mask1 > 0.7] = 1
            mask1 = mask1.astype(int)
            blobs = mask1

            skeleton_lee = skeletonize(blobs, method='lee')
            img = skeleton_lee

            (rows, cols) = np.nonzero(img)
            skel_coords = []

            for (r, c) in zip(rows, cols):
                (col_neigh, row_neigh) = np.meshgrid(
                    np.array([c - 1, c, c + 1]),
                    np.array([r - 1, r, r + 1])
                )
                col_neigh = col_neigh.astype('int')
                row_neigh = row_neigh.astype('int')

                # Clamp to image bounds
                row_neigh = np.clip(row_neigh, 0, img.shape[0] - 1)
                col_neigh = np.clip(col_neigh, 0, img.shape[1] - 1)

                pix_neighbourhood = img[row_neigh, col_neigh].ravel() != 0
                if np.sum(pix_neighbourhood) == 2:
                    skel_coords.append((r, c))

            if len(skel_coords) == 0:
                continue

            coord = np.asarray(skel_coords)
            img_centre = np.array([img.shape[0] / 2, img.shape[1] / 2])
            coords = sorted(coord, key=lambda point: np.linalg.norm(point - img_centre))
            coords = np.asarray(coords)

            # Store the closest skeleton endpoint to center
            centre_coord = coords[0, :]
            label = labels[i]
            if label < position.shape[0]:
                position[label, frame_index, 0:2] = centre_coord

    # ==================================================================
    # PUBLIC: Run inference on all surgery directories
    # ==================================================================
    def run_inference(self, surgery_dirs, images_root):
        """
        Run model inference on all images in each surgery directory.

        Args:
            surgery_dirs: list of surgery subdirectory names
            images_root: root path containing surgery subdirectories with images
        """
        self.model.eval()
        self.eval_predictions_per_surgery = {}
        self.preds_video_per_surgery = {}
        self.position_per_surgery = {}

        for surgery_name in surgery_dirs:
            img_dir = os.path.join(images_root, surgery_name)
            files = sorted([
                f for f in os.listdir(img_dir)
                if f.lower().endswith(('.jpg', '.png', '.jpeg'))
            ], key=natural_sort_key)

            surgery_eval_preds = {}
            surgery_preds_video = {}

            # Position array: shape (num_classes, num_frames+1, 2)
            # +1 because frame_id is 1-indexed
            position = np.zeros((self.num_classes, len(files) + 1, 2), dtype=int)

            print(f"\n--- [{surgery_name}] Running inference on {len(files)} images ---")

            for frame_id, fname in enumerate(files, start=1):
                img_path = os.path.join(img_dir, fname)
                img_pil = Image.open(img_path).convert("RGB")
                img_tensor = self.transform(img_pil)
                ih, iw = img_tensor.shape[1], img_tensor.shape[2]

                with torch.no_grad():
                    prediction = self.model([img_tensor.to(self.device)])

                # Filter by confidence
                preds = self.good_predictions(prediction, self.conf_threshold)

                # Apply NMS
                preds = self._apply_nms(preds)

                # Store in evaluator format: {filename: [boxes]}
                surgery_eval_preds[fname] = self._to_eval_format(preds)

                # Store in preds_video format: {frame_id: [boxes]}
                surgery_preds_video[frame_id] = self._to_preds_video_format(
                    preds, ih, iw, frame_id
                )

                # Compute skeleton endpoints for tool trajectory tracking
                self._compute_skeleton_position(preds, frame_id, position)

            self.eval_predictions_per_surgery[surgery_name] = surgery_eval_preds
            self.preds_video_per_surgery[surgery_name] = surgery_preds_video
            self.position_per_surgery[surgery_name] = position

            total_eval = sum(len(v) for v in surgery_eval_preds.values())
            print(f"  [{surgery_name}] {len(files)} frames → {total_eval} detections")

        print(f"\nInference complete for {len(surgery_dirs)} surgeries.")

    # ==================================================================
    # Accessors
    # ==================================================================
    def get_combined_eval_predictions(self):
        """
        Return a flat dict combining all surgeries for evaluation.
        Keys are prefixed with surgery name (matches gt_extractor.get_combined()).
        """
        combined = {}
        for surgery_name, preds in self.eval_predictions_per_surgery.items():
            for fname, boxes in preds.items():
                combined[f"{surgery_name}/{fname}"] = boxes
        return combined

    def get_preds_video(self, surgery_name):
        """Get preds_video dict for a specific surgery (for post-processing)."""
        return self.preds_video_per_surgery.get(surgery_name, {})

    def get_eval_predictions(self, surgery_name):
        """Get evaluator-format predictions for a specific surgery."""
        return self.eval_predictions_per_surgery.get(surgery_name, {})

    def get_position(self, surgery_name):
        """Get position array for a specific surgery (for trajectory analysis)."""
        return self.position_per_surgery.get(surgery_name, None)

    # ==================================================================
    # Save / Load
    # ==================================================================
    def save_all(self, output_dir):
        """Save per-surgery preds_video (pickle), position arrays (.npy), and eval format (.json)."""
        os.makedirs(output_dir, exist_ok=True)
        import json

        for surgery_name in self.preds_video_per_surgery:
            safe_name = surgery_name.replace(' ', '_').replace('/', '_')

            # Save preds_video as pickle
            preds_video = self.preds_video_per_surgery[surgery_name]
            pkl_path = os.path.join(output_dir, f"preds_video_{safe_name}.pkl")
            with open(pkl_path, 'wb') as f:
                no_frames = len(preds_video)
                pickle.dump((no_frames, preds_video), f)
            print(f"  📁 Saved: {pkl_path} ({len(preds_video)} frames)")

            # Save position array as .npy
            if surgery_name in self.position_per_surgery:
                npy_path = os.path.join(output_dir, f"position_{safe_name}.npy")
                np.save(npy_path, self.position_per_surgery[surgery_name])
                print(f"  📁 Saved: {npy_path} (shape: {self.position_per_surgery[surgery_name].shape})")

            # Save eval predictions as .json
            if surgery_name in self.eval_predictions_per_surgery:
                json_path = os.path.join(output_dir, f"eval_preds_{safe_name}.json")
                with open(json_path, 'w') as f:
                    json.dump(self.eval_predictions_per_surgery[surgery_name], f)

        print(f"All baseline predictions saved to: {output_dir}")

    def load_all(self, input_dir, surgery_dirs, images_root):
        """Load all saved inference data. Reconstructs eval format if JSON is missing."""
        import json
        self.eval_predictions_per_surgery = {}
        self.preds_video_per_surgery = {}
        self.position_per_surgery = {}

        loaded_count = 0
        for surgery_name in surgery_dirs:
            safe_name = surgery_name.replace(' ', '_').replace('/', '_')

            pkl_path = os.path.join(input_dir, f"preds_video_{safe_name}.pkl")
            npy_path = os.path.join(input_dir, f"position_{safe_name}.npy")
            json_path = os.path.join(input_dir, f"eval_preds_{safe_name}.json")

            if os.path.exists(pkl_path):
                # Load preds_video
                with open(pkl_path, 'rb') as f:
                    _, preds_video = pickle.load(f)
                self.preds_video_per_surgery[surgery_name] = preds_video

                # Load or reconstruct eval predictions
                if os.path.exists(json_path):
                    with open(json_path, 'r') as f:
                        self.eval_predictions_per_surgery[surgery_name] = json.load(f)
                else:
                    # Reconstruct from preds_video (backwards compatibility)
                    img_dir = os.path.join(images_root, surgery_name)
                    file_list = sorted([
                        f for f in os.listdir(img_dir)
                        if f.lower().endswith(('.jpg', '.png', '.jpeg'))
                    ], key=natural_sort_key)

                    eval_preds = {}
                    for frame_id, detections in preds_video.items():
                        if frame_id < 1 or frame_id > len(file_list):
                            continue
                        fname = file_list[frame_id - 1]

                        frame_preds = []
                        for det in detections:
                            scores = det.get('scores', None)
                            if scores is None:
                                continue
                            label = int(np.argmax(scores))
                            if label == 0:
                                continue  # Skip background
                            score = float(scores[label])
                            frame_preds.append({
                                'bbox': list(det['bbox']),
                                'label': label,
                                'score': score
                            })
                        eval_preds[fname] = frame_preds
                    self.eval_predictions_per_surgery[surgery_name] = eval_preds

                # Load position if exists
                if os.path.exists(npy_path):
                    self.position_per_surgery[surgery_name] = np.load(npy_path)

                loaded_count += 1
            else:
                print(f"⚠️ Missing saved data for {surgery_name} in {input_dir}")

        print(f"\n✅ Successfully loaded baseline data for {loaded_count} surgeries.")

    @staticmethod
    def load_preds_video(pkl_path):
        """Load a single surgery's preds_video from a pickle file."""
        with open(pkl_path, 'rb') as f:
            no_frames, preds_video = pickle.load(f)
        print(f"Loaded {no_frames} frames from: {pkl_path}")
        return preds_video

    @staticmethod
    def load_position(npy_path):
        """Load a single surgery's position array from a .npy file."""
        position = np.load(npy_path)
        print(f"Loaded position array (shape: {position.shape}) from: {npy_path}")
        return position

In [ ]:
# ================================================================
# Run baseline Inference
# ================================================================

BASELINE_OUTPUT_DIR = "/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/baseline_predictions"

predictor = BaselinePredictor(
    model=model,
    device=device,
    num_classes=6,           # 0=background + 5 tools
    conf_threshold=0.3,
    mask_iou_threshold=0.2
)

# ---------------------------------------------------------
# OPTION A: Run Inference (Takes Time)
# ---------------------------------------------------------
# predictor.run_inference(
#     surgery_dirs=gt_extractor.surgery_dirs,
#     images_root=IMAGES_ROOT
# )

# # Save all predictions (so you can load them next time)
# predictor.save_all(BASELINE_OUTPUT_DIR)


# ---------------------------------------------------------
# OPTION B: Load from Disk (Instant - Skip Option A)
# ---------------------------------------------------------
predictor.load_all(BASELINE_OUTPUT_DIR, gt_extractor.surgery_dirs, IMAGES_ROOT)

# Get combined predictions for evaluation
raw_predictions = predictor.get_combined_eval_predictions()
print(f"\nTotal combined predictions: {sum(len(v) for v in raw_predictions.values())}")


# To load position
# position_array = BaselinePredictor.load_position("path/to/position_surgery_1.npy")

In [ ]:
# ================================================================
# Run baseline Inference (Without any ground truth)
# ================================================================

import os

# 1. Define your new directory path
NEW_IMAGES_ROOT = "/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/NewImages"

# ---------------------------------------------------------
# OPTION A: Run Inference (Takes Time)
# ---------------------------------------------------------

# # ... OR, if you want it to automatically find all folders in that directory:
folders_to_run = [d for d in os.listdir(NEW_IMAGES_ROOT) if os.path.isdir(os.path.join(NEW_IMAGES_ROOT, d))]


# # 3. Run inference! (Notice we don't use gt_extractor here at all)
# predictor.run_inference(
#     surgery_dirs=folders_to_run,
#     images_root=NEW_IMAGES_ROOT
# )

# # 4. Save the predictions
# predictor.save_all(BASELINE_OUTPUT_DIR)

# ---------------------------------------------------------
# OPTION B: Load from Disk (Instant - Skip Option A)
# ---------------------------------------------------------
predictor.load_all(BASELINE_OUTPUT_DIR, folders_to_run, NEW_IMAGES_ROOT)

# Get combined predictions for evaluation
raw_predictions = predictor.get_combined_eval_predictions()
print(f"\nTotal combined predictions: {sum(len(v) for v in raw_predictions.values())}")


In [ ]:
# ================================================================
# Use the visualizer (WITHOUT Ground Truth)
# ================================================================

# 1. Explicitly set your surgery name
surgery = "surgery_10"

# 2. Recreate the visualizer pointing to your NEW image folder!
viz = DetectionVisualizer(
    label_to_name=LABEL_TO_NAME,
    images_root=NEW_IMAGES_ROOT  # Use the variable from the previous step
)

# 3. Get the predictions you just ran
preds_video = predictor.get_preds_video(surgery)


# --- Choose how you want to visualize ---

# Example 1: Show a single frame (Notice we removed `gt=gt`)
# viz.show_frame(surgery, preds_video, frame_id=18)

# Example 2: Show a grid of frames
# viz.show_grid(surgery, preds_video, frame_ids=[1, 5, 10, 15])

# Example 3: Show a range of frames
viz.show_range(surgery, preds_video)


In [ ]:
# ================================================================
# Use the visualizer
# ================================================================


# Example 1: Show a single frame with predictions + ground truth overlay
surgery = gt_extractor.surgery_dirs[0]  # Pick (idx+1)th surgery
preds_video = predictor.get_preds_video(surgery)
gt = gt_extractor.get_surgery_gt(surgery)

# viz.show_frame(surgery, preds_video, frame_id=18, gt=gt)

# Example 2: Show a grid of frames
# viz.show_grid(surgery, preds_video, frame_ids=[1, 5, 10, 15], gt=gt)

# Example 3: Show all frames for a surgery
# viz.show_range(surgery, preds_video, gt=gt)

# Without ground truth
viz.show_range(surgery, preds_video)

# Example 4: Compare baseline vs post-processed (after running DSU/gap-fill)
# viz.compare(surgery, baseline_pv, processed_pv, frame_id=5,
#             titles=["Raw Model", "DSU + Gap Fill"], gt=gt)

In [ ]:
# ================================================================
# Evaluate Raw Model (combined ground truth from all surgeries)
# ================================================================

ground_truth_combined = gt_extractor.get_combined()
results_raw = evaluator.evaluate("Raw Model", raw_predictions, ground_truth_combined)

In [ ]:
# ================================================================
# DSU Label Corrector Class
# ================================================================


import copy
import pickle
import os
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
from collections import Counter
from skimage.metrics import structural_similarity as ssim


class DSULabelCorrector:
    """
    DSU-based label correction for temporal consistency.

    Pipeline: Group similar frames → Merge overlapping bboxes → Majority vote labels

    Usage:
        dsu_corrector = DSULabelCorrector(IMAGES_ROOT)

        # Apply to all surgeries from predictor:
        dsu_corrector.apply_all(predictor)

        # Or apply to a single surgery:
        corrected_pv = dsu_corrector.apply("surgery_1", preds_video)

        # Save checkpoint (starting point for gap-fill methods):
        dsu_corrector.save_checkpoint(output_dir)

        # Visualize corrected predictions:
        viz.show_frame("surgery_1", dsu_corrector.get_corrected("surgery_1"), frame_id=1)
    """

    def __init__(self, images_root, ssim_threshold=0.75, overlap_threshold=0.5):
        """
        Args:
            images_root: root directory containing surgery subdirectories with images
            ssim_threshold: SSIM threshold for grouping similar frames (default 0.75)
            overlap_threshold: bbox overlap threshold for merging (default 0.5)
        """
        self.images_root = images_root
        self.ssim_threshold = ssim_threshold
        self.overlap_threshold = overlap_threshold
        self.transform = T.Compose([T.ToTensor()])

        # Results: {surgery_name: corrected_preds_video}
        self.corrected_per_surgery = {}

    # ==================================================================
    # DSU Operations (Disjoint Set Union with path compression)
    # ==================================================================
    @staticmethod
    def _get_parent(dsu, x):
        """Find root with path compression."""
        if dsu[x] == x:
            return x
        dsu[x] = DSULabelCorrector._get_parent(dsu, dsu[x])
        return dsu[x]

    @staticmethod
    def _union(dsu, adj, x, y):
        """Union two elements and update adjacency list."""
        x_set = DSULabelCorrector._get_parent(dsu, x)
        y_set = DSULabelCorrector._get_parent(dsu, y)
        if x_set == y_set:
            return
        adj[x_set].append(y_set)
        adj[y_set].append(x_set)
        dsu[y_set] = x_set

    # ==================================================================
    # SSIM Computation
    # ==================================================================
    @staticmethod
    def _compute_ssim(img1_tensor, img2_tensor):
        """
        Compute SSIM between two image tensors of shape (3, H, W).
        Values should be in [0, 1] (from ToTensor()).
        Returns: float in [-1, 1] (higher = more similar).
        """
        img1_gray = torch.mean(img1_tensor, dim=0).cpu().numpy()
        img2_gray = torch.mean(img2_tensor, dim=0).cpu().numpy()
        score, _ = ssim(img1_gray, img2_gray, data_range=1.0, full=True)
        return score

    # ==================================================================
    # Bbox Overlap Check
    # ==================================================================
    def _check_overlap(self, bbox1, bbox2):
        """
        Check if two bboxes overlap significantly.
        bbox format: (x, y, w, h).
        Uses Dice-like overlap: 2 * intersection / (area1 + area2).
        """
        x1, y1, w1, h1 = bbox1
        x2, y2, w2, h2 = bbox2

        xA = max(x1, x2)
        yA = max(y1, y2)
        xB = min(x1 + w1, x2 + w2)
        yB = min(y1 + h1, y2 + h2)

        inter_area = max(0, xB - xA) * max(0, yB - yA)
        total_area = w1 * h1 + w2 * h2

        if total_area == 0:
            return False

        return (2 * inter_area) / total_area >= self.overlap_threshold

    # ==================================================================
    # Helper: Collect all labels in a bbox cluster via DFS
    # ==================================================================
    @staticmethod
    def _collect_cluster_labels(start, bbox_adj, bbox_scores):
        """DFS to collect all score tuples in a bbox cluster."""
        stack = [start]
        seen = set()
        labels = []
        while stack:
            curr = stack.pop()
            if curr in seen:
                continue
            seen.add(curr)
            labels.append(bbox_scores[curr])
            for nei in bbox_adj[curr]:
                if nei not in seen:
                    stack.append(nei)
        return labels

    # ==================================================================
    # Helper: Load image as tensor for SSIM
    # ==================================================================
    def _load_image_tensor(self, surgery_name, frame_id, file_list):
        """
        Load image as tensor (3, H, W) with values in [0, 1].
        Args:
            surgery_name: surgery subdirectory name
            frame_id: 1-indexed frame number
            file_list: pre-sorted list of image filenames
        """
        if frame_id < 1 or frame_id > len(file_list):
            return None
        fname = file_list[frame_id - 1]
        img_path = os.path.join(self.images_root, surgery_name, fname)
        img = Image.open(img_path).convert("RGB")
        return self.transform(img)

    # ==================================================================
    # PUBLIC: Apply DSU label correction to a single surgery
    # ==================================================================
    def apply(self, surgery_name, preds_video):
        """
        Apply DSU label correction to a surgery's preds_video.

        Returns a NEW corrected preds_video (original is NOT modified).

        Args:
            surgery_name: surgery subdirectory name
            preds_video: {frame_id: [detections]} dict
        Returns:
            corrected preds_video dict (deep copy)
        """
        corrected = copy.deepcopy(preds_video)
        no_frames = max(corrected.keys()) if corrected else 0

        if no_frames == 0:
            print(f"  [{surgery_name}] No frames to process.")
            return corrected

        print(f"  [{surgery_name}] Applying DSU on {no_frames} frames...")

        # Get sorted file list once
        img_dir = os.path.join(self.images_root, surgery_name)
        file_list = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith(('.jpg', '.png', '.jpeg'))
        ], key=natural_sort_key)

        # =============================================================
        # PHASE 1: Frame-level DSU (group similar consecutive frames)
        # =============================================================
        frame_dsu = {i: i for i in range(1, no_frames + 1)}
        frame_adj = {i: [] for i in range(1, no_frames + 1)}

        # Cache images to avoid re-loading
        image_cache = {}

        for i in range(2, no_frames + 1):
            if i not in image_cache:
                image_cache[i] = self._load_image_tensor(
                    surgery_name, i, file_list
                )

            for j in range(i - 1, 0, -1):
                if j not in image_cache:
                    image_cache[j] = self._load_image_tensor(
                        surgery_name, j, file_list
                    )

                if image_cache[i] is None or image_cache[j] is None:
                    continue

                sim = self._compute_ssim(image_cache[i], image_cache[j])
                if sim >= self.ssim_threshold:
                    self._union(frame_dsu, frame_adj, i, j)
                    break  # Only merge with most recent similar frame

        # Free image cache
        del image_cache

        # =============================================================
        # PHASE 2: Connected Component Processing
        # =============================================================
        for root_frame in range(1, no_frames + 1):
            # Only process component roots
            if frame_dsu[root_frame] != root_frame:
                continue

            # --- Collect all bboxes from all frames in this component ---
            bbox_dsu = {}
            bbox_adj = {}
            bbox_scores = {}  # {bbox_tuple: scores_tuple}

            stk = [root_frame]
            visited = set()

            while stk:
                curr = stk.pop()
                if curr in visited:
                    continue
                visited.add(curr)

                if curr in corrected:
                    for obj in corrected[curr]:
                        bbox = tuple(obj["bbox"])
                        bbox_dsu[bbox] = bbox
                        bbox_adj[bbox] = []
                        bbox_scores[bbox] = tuple(
                            obj["scores"].tolist()
                            if hasattr(obj["scores"], 'tolist')
                            else obj["scores"]
                        )

                for nei in frame_adj[curr]:
                    if nei not in visited:
                        stk.append(nei)

            if not bbox_dsu:
                continue

            # --- Merge overlapping bboxes ---
            bboxes = list(bbox_dsu.keys())
            for i1 in range(len(bboxes)):
                for i2 in range(i1):
                    if self._check_overlap(bboxes[i1], bboxes[i2]):
                        self._union(bbox_dsu, bbox_adj, bboxes[i1], bboxes[i2])

            # --- Majority vote per bbox cluster ---
            bbox_majority = {}
            for bbox in bbox_dsu:
                if bbox_dsu[bbox] == bbox:  # Only process roots
                    labels = self._collect_cluster_labels(
                        bbox, bbox_adj, bbox_scores
                    )
                    # Most common label tuple wins
                    bbox_majority[bbox] = Counter(labels).most_common(1)[0][0]

            # --- Update labels back to all frames in this component ---
            stk = [root_frame]
            visited = set()

            while stk:
                curr = stk.pop()
                if curr in visited:
                    continue
                visited.add(curr)

                if curr in corrected:
                    for obj in corrected[curr]:
                        bbox = tuple(obj["bbox"])
                        root = self._get_parent(bbox_dsu, bbox)
                        # Restore as numpy array for consistency
                        obj["scores"] = np.array(bbox_majority[root])

                for nei in frame_adj[curr]:
                    if nei not in visited:
                        stk.append(nei)

        self.corrected_per_surgery[surgery_name] = corrected
        print(f"  [{surgery_name}] DSU label correction complete.")
        return corrected

    # ==================================================================
    # PUBLIC: Apply DSU to all surgeries from a BaselinePredictor
    # ==================================================================
    def apply_all(self, predictor):
        """
        Apply DSU to all surgeries stored in a BaselinePredictor.

        Args:
            predictor: BaselinePredictor instance (after run_inference)
        """
        for surgery_name in predictor.preds_video_per_surgery:
            preds_video = predictor.get_preds_video(surgery_name)
            self.apply(surgery_name, preds_video)
        print(f"\n✅ DSU applied to {len(self.corrected_per_surgery)} surgeries.")

    # ==================================================================
    # Accessors
    # ==================================================================
    def get_corrected(self, surgery_name):
        """Get corrected preds_video for a specific surgery."""
        return self.corrected_per_surgery.get(surgery_name, {})

    def get_all_corrected(self):
        """Get all corrected preds_video as {surgery_name: preds_video}."""
        return self.corrected_per_surgery


    def get_eval_predictions(self):
        """
        Get combined evaluator-format predictions for the DSU checkpoint.
        Returns: {surgery_name/filename: [{'bbox', 'label', 'score'}]}
        Keys match gt_extractor.get_combined() format.
        """
        combined = {}
        for surgery_name, pv in self.corrected_per_surgery.items():
            # Get sorted file list for this surgery
            img_dir = os.path.join(self.images_root, surgery_name)
            file_list = sorted([
                f for f in os.listdir(img_dir)
                if f.lower().endswith(('.jpg', '.png', '.jpeg'))
            ], key=natural_sort_key)

            for frame_id, detections in pv.items():
                if frame_id < 1 or frame_id > len(file_list):
                    continue
                fname = file_list[frame_id - 1]
                key = f"{surgery_name}/{fname}"

                frame_preds = []
                for det in detections:
                    scores = det.get('scores', None)
                    if scores is None:
                        continue
                    label = int(np.argmax(scores))
                    if label == 0:
                        continue  # Skip background
                    score = float(scores[label])
                    frame_preds.append({
                        'bbox': list(det['bbox']),
                        'label': label,
                        'score': score
                    })
                combined[key] = frame_preds

        return combined


    # ==================================================================
    # Save / Load checkpoint
    # ==================================================================
    def save_checkpoint(self, output_dir):
        """
        Save post-DSU preds_video for each surgery as pickle files.
        These serve as the starting point for all gap-filling methods.
        """
        os.makedirs(output_dir, exist_ok=True)

        for surgery_name, preds_video in self.corrected_per_surgery.items():
            safe_name = surgery_name.replace(' ', '_').replace('/', '_')
            pkl_path = os.path.join(output_dir, f"post_dsu_{safe_name}.pkl")

            with open(pkl_path, 'wb') as f:
                no_frames = len(preds_video)
                pickle.dump((no_frames, preds_video), f)
            print(f"  📁 Saved: {pkl_path} ({no_frames} frames)")

        print(f"DSU checkpoint saved to: {output_dir}")

    @staticmethod
    def load_checkpoint(pkl_path):
        """Load a post-DSU preds_video checkpoint from a pickle file."""
        with open(pkl_path, 'rb') as f:
            no_frames, preds_video = pickle.load(f)
        print(f"Loaded {no_frames} frames from: {pkl_path}")
        return preds_video

In [ ]:
# ================================================================
# APPLY DSU LABEL CORRECTION
# ================================================================

dsu_corrector = DSULabelCorrector(
    images_root=NEW_IMAGES_ROOT,
    ssim_threshold=0.75,
    overlap_threshold=0.5
)

# Apply DSU to all surgeries
dsu_corrector.apply_all(predictor)

# Save checkpoint (each gap-fill method loads from here)
DSU_CHECKPOINT_DIR = "/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/post_dsu_checkpoint"
dsu_corrector.save_checkpoint(DSU_CHECKPOINT_DIR)

In [ ]:
# ===============================================================================
# VISUALIZE: Compare baseline vs DSU-corrected (Without ground truth)
# ===============================================================================

surgery = "surgery_10"
baseline_pv = predictor.get_preds_video(surgery)
corrected_pv = dsu_corrector.get_corrected(surgery)

# Find out how many frames there are in total
total_frames = max(baseline_pv.keys()) if baseline_pv else 0

print(f"Comparing {total_frames} frames side-by-side...")

# Loop through every single frame and compare it
for frame_id in range(1, total_frames + 1):
    viz.compare(
        surgery_name=surgery,
        preds_video_1=baseline_pv,
        preds_video_2=corrected_pv,
        frame_id=frame_id,
        titles=["Raw Model", "After DSU"]
    )


In [ ]:
# ================================================================
# VISUALIZE: Compare baseline vs DSU-corrected (optional)
# ================================================================

surgery = gt_extractor.surgery_dirs[0]  # Pick first surgery
baseline_pv = predictor.get_preds_video(surgery)
corrected_pv = dsu_corrector.get_corrected(surgery)
gt = gt_extractor.get_surgery_gt(surgery)


# After DSU:
viz.show_frame(surgery, dsu_corrector.get_corrected(surgery), frame_id=5, gt=gt)

# Side-by-side comparison
viz.compare(surgery, baseline_pv, corrected_pv, frame_id=1,
            titles=["Raw Model", "After DSU"], gt=gt)

In [ ]:
!pip install filterpy

In [ ]:
import copy
import pickle
import os
import numpy as np
from PIL import Image

# Optional imports for SORT methods (Methods 6 & 7)
# Install with: !pip install filterpy
try:
    from filterpy.kalman import KalmanFilter as _KalmanFilter
    from scipy.optimize import linear_sum_assignment as _linear_sum_assignment
    _SORT_AVAILABLE = True
except ImportError:
    _SORT_AVAILABLE = False
    print("⚠️ filterpy/scipy not installed. SORT methods (6 & 7) unavailable.")
    print("   Install with: !pip install filterpy")


# ==================================================================
# Helper: Kalman-based Tool Tracker (for SORT - Method 6)
# ==================================================================
class _ToolTracker:
    """Kalman Filter tracker for a single tool. Used by SORT (Method 6)."""

    def __init__(self, bbox, label):
        self.kf = _KalmanFilter(dim_x=7, dim_z=4)
        # State: [cx, cy, area, aspect_ratio, dx, dy, d_area]
        self.kf.F = np.array([
            [1, 0, 0, 0, 1, 0, 0],
            [0, 1, 0, 0, 0, 1, 0],
            [0, 0, 1, 0, 0, 0, 1],
            [0, 0, 0, 1, 0, 0, 0],
            [0, 0, 0, 0, 1, 0, 0],
            [0, 0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 1]
        ])
        self.kf.H = np.array([
            [1, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0, 0],
            [0, 0, 1, 0, 0, 0, 0],
            [0, 0, 0, 1, 0, 0, 0]
        ])
        self.kf.R[2:, 2:] *= 10.
        self.kf.P[4:, 4:] *= 1000.
        self.kf.P *= 10.
        self.kf.Q[-1, -1] *= 0.01
        self.kf.Q[4:, 4:] *= 0.01

        self.label = label
        self.time_since_update = 0
        self.history = []
        self.ghost_tracks = []

        x, y, w, h = bbox
        self.kf.x[:4] = np.array([
            [x + w / 2], [y + h / 2],
            [w * h], [w / float(h + 1e-5)]
        ])

    def update(self, bbox):
        self.time_since_update = 0
        self.history = []
        x, y, w, h = bbox
        self.kf.update(np.array([
            [x + w / 2], [y + h / 2],
            [w * h], [w / float(h + 1e-5)]
        ]))

    def predict(self):
        if (self.kf.x[6] + self.kf.x[2]) <= 0:
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.time_since_update += 1

        cx, cy, s, r = self.kf.x[:4].flatten()
        s = max(1e-5, s)  # Guard against negative area
        r = max(1e-5, r)
        w = np.sqrt(s * r)
        h = s / (w + 1e-5)
        x = cx - w / 2
        y = cy - h / 2
        self.history.append([x, y, w, h])
        return [x, y, w, h]


# ==================================================================
# Helper: Kalman-based Tool Tracker (for Hybrid - Method 7)
# ==================================================================
class _ToolTrackerHybrid:
    """Kalman Filter tracker that remembers last actual detection.
    Used by Hybrid SORT + Interpolation (Method 7)."""

    def __init__(self, bbox, label, fid):
        self.kf = _KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([
            [1, 0, 0, 0, 1, 0, 0],
            [0, 1, 0, 0, 0, 1, 0],
            [0, 0, 1, 0, 0, 0, 1],
            [0, 0, 0, 1, 0, 0, 0],
            [0, 0, 0, 0, 1, 0, 0],
            [0, 0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 0, 1]
        ])
        self.kf.H = np.array([
            [1, 0, 0, 0, 0, 0, 0],
            [0, 1, 0, 0, 0, 0, 0],
            [0, 0, 1, 0, 0, 0, 0],
            [0, 0, 0, 1, 0, 0, 0]
        ])
        self.kf.R[2:, 2:] *= 10.
        self.kf.P[4:, 4:] *= 1000.
        self.kf.P *= 10.
        self.kf.Q[-1, -1] *= 0.01
        self.kf.Q[4:, 4:] *= 0.01

        self.label = label
        self.time_since_update = 0
        self.last_actual_bbox = np.array(bbox)
        self.last_actual_fid = fid

        x, y, w, h = bbox
        self.kf.x[:4] = np.array([
            [x + w / 2], [y + h / 2],
            [w * h], [w / float(h + 1e-5)]
        ])

    def update(self, bbox, fid):
        self.time_since_update = 0
        self.last_actual_bbox = np.array(bbox)
        self.last_actual_fid = fid
        x, y, w, h = bbox
        self.kf.update(np.array([
            [x + w / 2], [y + h / 2],
            [w * h], [w / float(h + 1e-5)]
        ]))

    def predict(self):
        if (self.kf.x[6] + self.kf.x[2]) <= 0:
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.time_since_update += 1

        cx, cy, s, r = self.kf.x[:4].flatten()
        s = max(1e-5, s)
        r = max(1e-5, r)
        w = np.sqrt(s * r)
        h = s / (w + 1e-5)
        x = cx - w / 2
        y = cy - h / 2
        return [x, y, w, h]


class GapFillingProcessor:
    """
    Container for all gap-filling post-processing methods.

    Each method follows the sandwich pipeline:
      DSU Checkpoint → Deep Copy → Gap Fill → DSU (2nd pass) → Store

    Usage:
        processor = GapFillingProcessor(dsu_corrector, IMAGES_ROOT)

        # Run a method (full sandwich automatically):
        processor.run_method("DSU + Gap Fill ±2 + DSU",
                             processor.gap_fill_neighbors, window=2)

        # Get results for evaluation:
        eval_preds = processor.get_eval_predictions("DSU + Gap Fill ±2 + DSU")
        evaluator.evaluate("DSU + Gap Fill ±2 + DSU", eval_preds, gt_combined)

        # Visualize:
        pv = processor.get_preds_video("DSU + Gap Fill ±2 + DSU", "surgery_1")
        viz.show_frame("surgery_1", pv, frame_id=5)
    """

    def __init__(self, dsu_corrector, images_root, num_classes=6):
        """
        Args:
            dsu_corrector: DSULabelCorrector instance (with checkpoint already applied)
            images_root: root directory containing surgery subdirectories
            num_classes: number of classes including background (default 6)
        """
        self.dsu_corrector = dsu_corrector
        self.images_root = images_root
        self.num_classes = num_classes

        # For the second DSU pass (sandwich pipeline)
        # Uses same parameters as the first DSU but stores results separately
        self._second_dsu = DSULabelCorrector(
            images_root=images_root,
            ssim_threshold=dsu_corrector.ssim_threshold,
            overlap_threshold=dsu_corrector.overlap_threshold
        )

        # {method_name: {surgery_name: preds_video_after_sandwich}}
        self.results = {}

    # ==================================================================
    # Shared Helper: Get tools detected in a frame
    # ==================================================================
    @staticmethod
    def _get_tools_in_frame(preds_video, fid):
        """
        Get a dict of {label_id: detection_entry} for tools in a frame.
        Skips background (label 0).
        """
        tools = {}
        if fid not in preds_video:
            return tools
        for entry in preds_video[fid]:
            if 'scores' in entry:
                lbl = int(np.argmax(entry['scores']))
                if lbl != 0:
                    tools[lbl] = entry
        return tools

    # ==================================================================
    # Pipeline: Run full sandwich for a method
    # ==================================================================
    def run_method(self, method_name, method_func, **method_kwargs):
        """
        Run a gap-filling method on all surgeries with the full sandwich:
        DSU Checkpoint → Deep Copy → Gap Fill → DSU (2nd pass) → Store

        Args:
            method_name: display name (e.g., "DSU + Gap Fill ±2 + DSU")
            method_func: the gap-filling function (must accept preds_video
                         as first arg and return modified preds_video)
            **method_kwargs: additional kwargs passed to the gap-fill function
        """
        print(f"\n{'=' * 65}")
        print(f"  Running: {method_name}")
        print(f"{'=' * 65}")

        method_results = {}

        for surgery_name in self.dsu_corrector.corrected_per_surgery:
            print(f"\n  [{surgery_name}]")

            # Step 1: Deep copy the DSU checkpoint (never modify original)
            checkpoint_pv = self.dsu_corrector.get_corrected(surgery_name)
            pv = copy.deepcopy(checkpoint_pv)

            # Step 2: Apply gap-filling method
            print(f"    Applying gap filling...")
            self._current_surgery = surgery_name  # Methods can access this
            pv = method_func(pv, **method_kwargs)

            # Step 3: Apply DSU again (second pass for label consistency)
            print(f"    Applying DSU (2nd pass)...")
            pv = self._second_dsu.apply(surgery_name, pv)

            method_results[surgery_name] = pv

        self.results[method_name] = method_results
        print(f"\n✅ {method_name} — complete for {len(method_results)} surgeries.\n")

    # ==================================================================
    # Accessors
    # ==================================================================
    def get_preds_video(self, method_name, surgery_name):
        """Get preds_video for a specific method + surgery (for visualization)."""
        if method_name not in self.results:
            print(f"⚠️ Method '{method_name}' not found.")
            return {}
        return self.results[method_name].get(surgery_name, {})

    def get_eval_predictions(self, method_name):
        """
        Get combined evaluator-format predictions for a method.
        Returns: {surgery_name/filename: [{'bbox', 'label', 'score'}]}
        Keys match gt_extractor.get_combined() format.
        """
        if method_name not in self.results:
            print(f"⚠️ Method '{method_name}' not found.")
            return {}

        combined = {}

        for surgery_name, pv in self.results[method_name].items():
            # Get sorted file list for this surgery
            img_dir = os.path.join(self.images_root, surgery_name)
            file_list = sorted([
                f for f in os.listdir(img_dir)
                if f.lower().endswith(('.jpg', '.png', '.jpeg'))
            ], key=natural_sort_key)

            for frame_id, detections in pv.items():
                if frame_id < 1 or frame_id > len(file_list):
                    continue
                fname = file_list[frame_id - 1]
                key = f"{surgery_name}/{fname}"

                frame_preds = []
                for det in detections:
                    scores = det.get('scores', None)
                    if scores is None:
                        continue
                    label = int(np.argmax(scores))
                    if label == 0:
                        continue  # Skip background
                    score = float(scores[label])
                    frame_preds.append({
                        'bbox': list(det['bbox']),
                        'label': label,
                        'score': score
                    })
                combined[key] = frame_preds

        return combined

    def list_methods(self):
        """Print all methods that have been run."""
        if not self.results:
            print("No methods have been run yet.")
            return
        print("Completed methods:")
        for name in self.results:
            n_surgeries = len(self.results[name])
            print(f"  • {name} ({n_surgeries} surgeries)")

    # ==================================================================
    # Save / Load
    # ==================================================================
    def save_results(self, output_dir, method_name=None):
        """
        Save preds_video results to pickle files.
        Args:
            output_dir: root output directory
            method_name: save only this method (or all if None)
        """
        os.makedirs(output_dir, exist_ok=True)

        methods = [method_name] if method_name else list(self.results.keys())

        for mname in methods:
            if mname not in self.results:
                print(f"⚠️ Method '{mname}' not found, skipping.")
                continue

            # Create subdirectory per method
            safe_method = (mname.replace(' ', '_').replace('/', '_')
                               .replace('±', 'pm').replace('+', 'plus'))
            method_dir = os.path.join(output_dir, safe_method)
            os.makedirs(method_dir, exist_ok=True)

            for surgery_name, pv in self.results[mname].items():
                safe_surgery = surgery_name.replace(' ', '_').replace('/', '_')
                pkl_path = os.path.join(
                    method_dir, f"preds_video_{safe_surgery}.pkl"
                )
                with open(pkl_path, 'wb') as f:
                    pickle.dump((len(pv), pv), f)
                print(f"  📁 Saved: {pkl_path} ({len(pv)} frames)")

        print(f"Results saved to: {output_dir}")

    @staticmethod
    def load_result(pkl_path):
        """Load a saved preds_video result."""
        with open(pkl_path, 'rb') as f:
            no_frames, preds_video = pickle.load(f)
        print(f"Loaded {no_frames} frames from: {pkl_path}")
        return preds_video

    # ==================================================================
    # ==================== GAP FILLING METHODS =========================
    # ==================================================================
    # Add new methods below. Each method must:
    #   1. Accept preds_video as the first argument
    #   2. Modify preds_video (it's already a deep copy)
    #   3. Return the modified preds_video
    # ==================================================================

    def gap_fill_neighbors(self, preds_video, window=2):
        """
        Method 1: Gap Fill with ±window neighbor frames.

        For each frame, looks at the closest ±window neighbor frames.
        If a tool is present in BOTH left and right windows but MISSING
        in the current frame, interpolates the bounding box (average of
        nearest left and right detections).

        Args:
            preds_video: {frame_id: [detections]} dict (will be modified)
            window: number of frames to look in each direction (default 2)
        Returns:
            modified preds_video
        """
        sorted_frame_ids = sorted(preds_video.keys())
        total_frames = len(sorted_frame_ids)
        count_fixed = 0

        for i in range(total_frames):
            curr_fid = sorted_frame_ids[i]
            tools_curr = self._get_tools_in_frame(preds_video, curr_fid)

            # --- Search LEFT window ---
            closest_left = {}
            for offset in range(1, window + 1):
                left_idx = i - offset
                if left_idx < 0:
                    break
                left_fid = sorted_frame_ids[left_idx]
                tools_left = self._get_tools_in_frame(preds_video, left_fid)
                for label, entry in tools_left.items():
                    if label not in closest_left:
                        closest_left[label] = entry

            # --- Search RIGHT window ---
            closest_right = {}
            for offset in range(1, window + 1):
                right_idx = i + offset
                if right_idx >= total_frames:
                    break
                right_fid = sorted_frame_ids[right_idx]
                tools_right = self._get_tools_in_frame(preds_video, right_fid)
                for label, entry in tools_right.items():
                    if label not in closest_right:
                        closest_right[label] = entry

            # --- Fill gaps: tools in BOTH windows but MISSING in current ---
            missed_tools = (set(closest_left.keys()) &
                            set(closest_right.keys()))

            for label in missed_tools:
                if label not in tools_curr:
                    prev_entry = closest_left[label]
                    next_entry = closest_right[label]

                    # Interpolate bounding box (average)
                    box_prev = np.array(prev_entry['bbox'])
                    box_next = np.array(next_entry['bbox'])
                    avg_box = (box_prev + box_next) / 2.0

                    # Compute center from interpolated box
                    avg_cx = avg_box[0] + (avg_box[2] / 2.0)
                    avg_cy = avg_box[1] + (avg_box[3] / 2.0)

                    # Create one-hot scores for the detected label
                    new_scores = np.zeros(self.num_classes, dtype=float)
                    new_scores[label] = 1.0

                    new_entry = {
                        'image_id': curr_fid,
                        'bbox': avg_box.tolist(),
                        'bbox_center': [float(avg_cx), float(avg_cy)],
                        'scores': new_scores
                    }

                    if curr_fid not in preds_video:
                        preds_video[curr_fid] = []

                    preds_video[curr_fid].append(new_entry)
                    count_fixed += 1

        print(f"    Gap Fill ±{window}: Rectified {count_fixed} missed detections.")
        return preds_video

    # ==================================================================
    # Shared Helper: Euclidean distance between bbox centers
    # ==================================================================
    @staticmethod
    def _box_center_distance(box1, box2):
        """Euclidean distance between centers of two [x, y, w, h] bboxes."""
        x1, y1, w1, h1 = box1
        x2, y2, w2, h2 = box2
        c1 = np.array([x1 + w1 / 2.0, y1 + h1 / 2.0])
        c2 = np.array([x2 + w2 / 2.0, y2 + h2 / 2.0])
        return np.linalg.norm(c1 - c2)

    # ==================================================================
    # Method 2: Advanced Temporal Interpolation (Kinematic)
    # ==================================================================
    def advanced_temporal_interp(self, preds_video, window=4,
                                 max_pixel_jump=150.0):
        """
        Method 2: Advanced Temporal Interpolation.

        Like Method 1, but with two key improvements:
        1. Distance validation: skips repairs where the tool jumped
           more than max_pixel_jump pixels between anchor frames.
        2. Weighted (kinematic) interpolation: bbox position is
           weighted by temporal distance to each anchor frame.
        3. Chain propagation: the frame cache is updated after each
           repair, so filling gap A can enable filling gap B.

        Args:
            preds_video: {frame_id: [detections]} dict (will be modified)
            window: frames to look in each direction (default 4)
            max_pixel_jump: max center distance between anchors (default 150)
        Returns:
            modified preds_video
        """
        sorted_fids = sorted(preds_video.keys())
        no_frames = max(sorted_fids) if sorted_fids else 0
        count_fixed = 0

        # Pre-compute cache: {fid: {label: entry}}
        cache = {}
        for fid in range(1, no_frames + 1):
            cache[fid] = {}
            if fid in preds_video:
                for entry in preds_video[fid]:
                    if 'scores' in entry:
                        lbl = int(np.argmax(entry['scores']))
                        if lbl != 0:
                            cache[fid][lbl] = entry

        # Iterate and repair
        for curr_fid in range(1, no_frames + 1):
            tools_curr = cache.get(curr_fid, {})

            # Search LEFT window
            closest_left = {}
            for offset in range(1, window + 1):
                left_fid = curr_fid - offset
                if left_fid < 1:
                    break
                for label, entry in cache.get(left_fid, {}).items():
                    if label not in closest_left:
                        closest_left[label] = (left_fid, entry)

            # Search RIGHT window
            closest_right = {}
            for offset in range(1, window + 1):
                right_fid = curr_fid + offset
                if right_fid > no_frames:
                    break
                for label, entry in cache.get(right_fid, {}).items():
                    if label not in closest_right:
                        closest_right[label] = (right_fid, entry)

            # Find missed tools
            missed = (set(closest_left.keys()) &
                      set(closest_right.keys()))

            for label in missed:
                if label not in tools_curr:
                    prev_fid, prev_entry = closest_left[label]
                    next_fid, next_entry = closest_right[label]

                    # Distance check: skip if tool jumped too far
                    dist = self._box_center_distance(
                        prev_entry['bbox'], next_entry['bbox']
                    )
                    if dist > max_pixel_jump:
                        continue

                    # Kinematic weighted interpolation
                    box_prev = np.array(prev_entry['bbox'])
                    box_next = np.array(next_entry['bbox'])
                    gap_size = next_fid - prev_fid
                    w1 = (next_fid - curr_fid) / float(gap_size)
                    w2 = (curr_fid - prev_fid) / float(gap_size)
                    weighted_box = (w1 * box_prev) + (w2 * box_next)

                    avg_cx = weighted_box[0] + (weighted_box[2] / 2.0)
                    avg_cy = weighted_box[1] + (weighted_box[3] / 2.0)

                    new_scores = np.zeros(self.num_classes, dtype=float)
                    new_scores[label] = 1.0

                    new_entry = {
                        'image_id': curr_fid,
                        'bbox': weighted_box.tolist(),
                        'bbox_center': [float(avg_cx), float(avg_cy)],
                        'scores': new_scores
                    }

                    if curr_fid not in preds_video:
                        preds_video[curr_fid] = []
                    preds_video[curr_fid].append(new_entry)

                    # Update cache for chain propagation
                    cache[curr_fid][label] = new_entry
                    count_fixed += 1

        print(f"    Advanced Temporal (window={window}, max_jump={max_pixel_jump}): "
              f"Rectified {count_fixed} missed detections.")
        return preds_video

    # ==================================================================
    # Method 3: Linear Interpolation (Paper Equation 2)
    # ==================================================================
    def linear_interp_paper(self, preds_video, max_gap=25):
        """
        Method 3: Paper-based Linear Interpolation.

        For each tool class, finds consecutive detection frames and fills
        all intermediate frames using linear interpolation:

            b_j = ((t_{i+1} - t_j) / (t_{i+1} - t_i)) * b_i
                + ((t_j - t_i) / (t_{i+1} - t_i)) * b_{i+1}

        Only bridges gaps up to max_gap frames (paper default: 25 = 1s @ 25fps).

        Args:
            preds_video: {frame_id: [detections]} dict (will be modified)
            max_gap: maximum gap size to bridge (default 25)
        Returns:
            modified preds_video
        """
        sorted_fids = sorted(preds_video.keys())
        no_frames = max(sorted_fids) if sorted_fids else 0
        count_fixed = 0

        # Iterate per tool class (skip background=0)
        for label in range(1, self.num_classes):

            # Find all frames where this tool is detected
            detected_fids = []
            for fid in range(1, no_frames + 1):
                if fid in preds_video:
                    for entry in preds_video[fid]:
                        if int(np.argmax(entry['scores'])) == label:
                            detected_fids.append(fid)
                            break

            # Bridge gaps between consecutive detections
            for idx in range(len(detected_fids) - 1):
                ti = detected_fids[idx]
                ti_next = detected_fids[idx + 1]
                gap_size = ti_next - ti

                if gap_size <= 1 or gap_size > max_gap:
                    continue

                # Find the bbox at ti and ti_next
                bi = None
                for entry in preds_video[ti]:
                    if int(np.argmax(entry['scores'])) == label:
                        bi = np.array(entry['bbox'])
                        break

                bi_next = None
                for entry in preds_video[ti_next]:
                    if int(np.argmax(entry['scores'])) == label:
                        bi_next = np.array(entry['bbox'])
                        break

                if bi is None or bi_next is None:
                    continue

                # Interpolate all frames in the gap
                for tj in range(ti + 1, ti_next):
                    w1 = (ti_next - tj) / float(gap_size)
                    w2 = (tj - ti) / float(gap_size)
                    bj = (w1 * bi) + (w2 * bi_next)

                    avg_cx = bj[0] + (bj[2] / 2.0)
                    avg_cy = bj[1] + (bj[3] / 2.0)

                    new_scores = np.zeros(self.num_classes, dtype=float)
                    new_scores[label] = 1.0

                    new_entry = {
                        'image_id': tj,
                        'bbox': bj.tolist(),
                        'bbox_center': [float(avg_cx), float(avg_cy)],
                        'scores': new_scores
                    }

                    if tj not in preds_video:
                        preds_video[tj] = []
                    preds_video[tj].append(new_entry)
                    count_fixed += 1

        print(f"    Linear Interp (max_gap={max_gap}): "
              f"Rectified {count_fixed} missed detections.")
        return preds_video

    # ==================================================================
    # Method 4: Advanced Linear Interpolation (Paper Eq. 2 + Distance)
    # ==================================================================
    def advanced_linear_interp(self, preds_video, max_gap=25,
                                max_pixel_jump=150.0):
        """
        Method 4: Paper-based Linear Interpolation with distance check.

        Same per-tool gap bridging as Method 3, but with a safety check:
        skips gaps where the tool center moved more than max_pixel_jump
        pixels between the two anchor frames.

        Args:
            preds_video: {frame_id: [detections]} dict (will be modified)
            max_gap: maximum gap size to bridge (default 25)
            max_pixel_jump: max center distance between anchors (default 150)
        Returns:
            modified preds_video
        """
        sorted_fids = sorted(preds_video.keys())
        no_frames = max(sorted_fids) if sorted_fids else 0
        count_fixed = 0

        # Iterate per tool class (skip background=0)
        for label in range(1, self.num_classes):

            # Find all frames where this tool is detected, with their bbox
            detected_tools = []
            for fid in range(1, no_frames + 1):
                if fid in preds_video:
                    for entry in preds_video[fid]:
                        if int(np.argmax(entry['scores'])) == label:
                            detected_tools.append(
                                (fid, np.array(entry['bbox']))
                            )
                            break

            # Bridge gaps between consecutive detections
            for idx in range(len(detected_tools) - 1):
                ti, bi = detected_tools[idx]
                ti_next, bi_next = detected_tools[idx + 1]
                gap_size = ti_next - ti

                if gap_size <= 1 or gap_size > max_gap:
                    continue

                # Distance check
                if self._box_center_distance(bi, bi_next) > max_pixel_jump:
                    continue

                # Interpolate all frames in the gap
                for tj in range(ti + 1, ti_next):
                    w1 = (ti_next - tj) / float(gap_size)
                    w2 = (tj - ti) / float(gap_size)
                    bj = (w1 * bi) + (w2 * bi_next)

                    avg_cx = bj[0] + (bj[2] / 2.0)
                    avg_cy = bj[1] + (bj[3] / 2.0)

                    new_scores = np.zeros(self.num_classes, dtype=float)
                    new_scores[label] = 1.0

                    new_entry = {
                        'image_id': tj,
                        'bbox': bj.tolist(),
                        'bbox_center': [float(avg_cx), float(avg_cy)],
                        'scores': new_scores
                    }

                    if tj not in preds_video:
                        preds_video[tj] = []
                    preds_video[tj].append(new_entry)
                    count_fixed += 1

        print(f"    Adv Linear Interp (max_gap={max_gap}, "
              f"max_jump={max_pixel_jump}): "
              f"Rectified {count_fixed} missed detections.")
        return preds_video

    # ==================================================================
    # Method 5: Template Matching (Visual Tracking)
    # ==================================================================
    def template_matching(self, preds_video, max_gap=25,
                          match_threshold=0.4):
        """
        Method 5: Template Matching based visual tracking.

        For each tool gap, extracts a grayscale template from the last
        detection frame and uses OpenCV's normalized cross-correlation
        to find the tool in each gap frame. Only accepts matches above
        the confidence threshold.

        Note: Uses self._current_surgery (set by run_method) to load
        images from the correct surgery directory.

        Args:
            preds_video: {frame_id: [detections]} dict (will be modified)
            max_gap: maximum gap size to bridge (default 25)
            match_threshold: minimum correlation score to accept (default 0.4)
        Returns:
            modified preds_video
        """
        import cv2

        surgery_name = self._current_surgery

        # Get sorted file list for image loading
        img_dir = os.path.join(self.images_root, surgery_name)
        file_list = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith(('.jpg', '.png', '.jpeg'))
        ], key=natural_sort_key)

        sorted_fids = sorted(preds_video.keys())
        no_frames = max(sorted_fids) if sorted_fids else 0
        count_fixed = 0
        count_skipped = 0

        def load_gray(frame_id):
            """Load a frame as grayscale uint8 numpy array."""
            if frame_id < 1 or frame_id > len(file_list):
                return None
            path = os.path.join(img_dir, file_list[frame_id - 1])
            img = np.array(Image.open(path).convert("RGB"))
            return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        # Iterate per tool class
        for label in range(1, self.num_classes):

            # Find detected frames for this tool
            detected_fids = []
            for fid in range(1, no_frames + 1):
                if fid in preds_video:
                    for entry in preds_video[fid]:
                        if int(np.argmax(entry['scores'])) == label:
                            detected_fids.append(fid)
                            break

            # Bridge gaps using template matching
            for idx in range(len(detected_fids) - 1):
                ti = detected_fids[idx]
                ti_next = detected_fids[idx + 1]
                gap_size = ti_next - ti

                if gap_size <= 1 or gap_size > max_gap:
                    continue

                # Find the bbox at the anchor frame
                bi = None
                for entry in preds_video[ti]:
                    if int(np.argmax(entry['scores'])) == label:
                        bi = np.array(entry['bbox']).astype(int)
                        break

                if bi is None:
                    continue

                # Load anchor frame and extract template
                img_gray_ti = load_gray(ti)
                if img_gray_ti is None:
                    continue

                img_h, img_w = img_gray_ti.shape
                x, y, w, h = bi
                x_start = max(0, x)
                y_start = max(0, y)
                x_end = min(img_w, x + w)
                y_end = min(img_h, y + h)

                template = img_gray_ti[y_start:y_end, x_start:x_end]

                if template.size == 0:
                    continue

                # Template must be smaller than target image
                if template.shape[0] >= img_h or template.shape[1] >= img_w:
                    continue

                # Search each gap frame
                for tj in range(ti + 1, ti_next):
                    img_gray_tj = load_gray(tj)
                    if img_gray_tj is None:
                        continue

                    res = cv2.matchTemplate(
                        img_gray_tj, template, cv2.TM_CCOEFF_NORMED
                    )
                    _, max_val, _, max_loc = cv2.minMaxLoc(res)

                    # Reject low-confidence matches
                    if max_val < match_threshold:
                        count_skipped += 1
                        continue

                    current_x, current_y = max_loc
                    avg_cx = current_x + (w / 2.0)
                    avg_cy = current_y + (h / 2.0)

                    new_scores = np.zeros(self.num_classes, dtype=float)
                    new_scores[label] = 1.0

                    new_entry = {
                        'image_id': tj,
                        'bbox': [float(current_x), float(current_y),
                                 float(w), float(h)],
                        'bbox_center': [float(avg_cx), float(avg_cy)],
                        'scores': new_scores
                    }

                    if tj not in preds_video:
                        preds_video[tj] = []
                    preds_video[tj].append(new_entry)
                    count_fixed += 1

        print(f"    Template Matching (max_gap={max_gap}, "
              f"threshold={match_threshold}): "
              f"Rectified {count_fixed}, skipped {count_skipped} "
              f"low-confidence matches.")
        return preds_video

    # ==================================================================
    # Shared Helper: IoU between two bboxes
    # ==================================================================
    @staticmethod
    def _iou_bbox(box1, box2):
        """IoU between two [x, y, w, h] bboxes."""
        x1, y1, w1, h1 = box1
        x2, y2, w2, h2 = box2
        xA = max(x1, x2)
        yA = max(y1, y2)
        xB = min(x1 + w1, x2 + w2)
        yB = min(y1 + h1, y2 + h2)
        inter = max(0, xB - xA) * max(0, yB - yA)
        area1 = w1 * h1
        area2 = w2 * h2
        return inter / float(area1 + area2 - inter + 1e-5)

    # ==================================================================
    # Shared Helper: IoU + IoM (Intersection over Minimum)
    # ==================================================================
    @staticmethod
    def _calculate_overlap_metrics(box1, box2):
        """Returns (IoU, IoM) for two [x, y, w, h] bboxes."""
        x1, y1, w1, h1 = box1
        x2, y2, w2, h2 = box2
        xA = max(x1, x2)
        yA = max(y1, y2)
        xB = min(x1 + w1, x2 + w2)
        yB = min(y1 + h1, y2 + h2)
        inter = max(0, xB - xA) * max(0, yB - yA)
        area1 = w1 * h1
        area2 = w2 * h2
        iou = inter / float(area1 + area2 - inter + 1e-5)
        iom = inter / float(min(area1, area2) + 1e-5)
        return iou, iom

    # ==================================================================
    # Method 6: SORT (Simple Online and Realtime Tracking)
    # ==================================================================
    def sort_tracking(self, preds_video, max_age=25, overlap_thresh=0.5):
        """
        Method 6: SORT-based tracking with Kalman Filters.

        Uses per-tool Kalman Filter trackers with Hungarian matching.
        When a tracker loses its detection, it stores ghost tracks
        (predicted positions). When re-detected, all ghost tracks
        are flushed as interpolated detections.

        Also applies intra-class NMS to remove duplicate detections.

        Requires: pip install filterpy

        NOTE: Unlike Methods 1-5 which modify preds_video in place,
        SORT rebuilds a fresh preds_video from tracked detections.
        The returned dict replaces the input entirely.

        Args:
            preds_video: {frame_id: [detections]} dict
            max_age: max frames a tracker survives without detection (default 25)
            overlap_thresh: IoU threshold for intra-class NMS (default 0.5)
        Returns:
            new preds_video built from tracked detections
        """
        if not _SORT_AVAILABLE:
            print("    ⚠️ SORT unavailable (filterpy not installed). Skipping.")
            return preds_video

        sorted_fids = sorted(preds_video.keys())
        no_frames = max(sorted_fids) if sorted_fids else 0

        repaired_preds = {i: [] for i in range(1, no_frames + 1)}
        trackers_by_label = {l: [] for l in range(1, self.num_classes)}
        count_fixed = 0
        duplicates_removed = 0

        for fid in range(1, no_frames + 1):
            current_detections = preds_video.get(fid, [])

            for label in trackers_by_label:
                raw_dets = [
                    d for d in current_detections
                    if int(np.argmax(d['scores'])) == label
                ]

                # Intra-class NMS
                dets = []
                for d in raw_dets:
                    is_dup = False
                    for fd in dets:
                        if self._iou_bbox(d['bbox'], fd['bbox']) > overlap_thresh:
                            is_dup = True
                            duplicates_removed += 1
                            # Keep the larger box
                            if (d['bbox'][2] * d['bbox'][3]) > \
                               (fd['bbox'][2] * fd['bbox'][3]):
                                fd['bbox'] = d['bbox']
                                fd['bbox_center'] = d.get('bbox_center',
                                    fd.get('bbox_center'))
                            break
                    if not is_dup:
                        dets.append(d)

                trks = trackers_by_label[label]

                # 1. Predict next locations
                predicted_boxes = [trk.predict() for trk in trks]

                # 2. Match using Center Distance + Hungarian
                matched = []
                unmatched_dets_set = set(range(len(dets)))
                unmatched_trks_set = set(range(len(trks)))

                if len(dets) > 0 and len(trks) > 0:
                    cost = np.zeros((len(dets), len(trks)))
                    for d_idx, d in enumerate(dets):
                        for t_idx, t_box in enumerate(predicted_boxes):
                            cost[d_idx, t_idx] = \
                                self._box_center_distance(d['bbox'], t_box)

                    row_ind, col_ind = _linear_sum_assignment(cost)
                    for r, c in zip(row_ind, col_ind):
                        if cost[r, c] < 150.0:
                            matched.append((r, c))
                            unmatched_dets_set.discard(r)
                            unmatched_trks_set.discard(c)

                unmatched_dets = list(unmatched_dets_set)

                # 3. Matched: flush ghost tracks + update
                for d_idx, t_idx in matched:
                    trk = trks[t_idx]

                    # Flush ghost tracks as interpolated detections
                    if len(trk.ghost_tracks) > 0:
                        for ghost_fid, ghost_box in trk.ghost_tracks:
                            x, y, w, h = ghost_box
                            new_scores = np.zeros(self.num_classes,
                                                  dtype=float)
                            new_scores[label] = 1.0
                            new_entry = {
                                'image_id': ghost_fid,
                                'bbox': [float(x), float(y),
                                         float(w), float(h)],
                                'bbox_center': [float(x + w / 2.0),
                                                float(y + h / 2.0)],
                                'scores': new_scores
                            }
                            repaired_preds[ghost_fid].append(new_entry)
                            count_fixed += 1
                        trk.ghost_tracks = []

                    trk.update(dets[d_idx]['bbox'])
                    repaired_preds[fid].append(dets[d_idx])

                # 4. Unmatched detections: new trackers
                for d_idx in unmatched_dets:
                    trks.append(
                        _ToolTracker(dets[d_idx]['bbox'], label)
                    )
                    repaired_preds[fid].append(dets[d_idx])

                # 5. Unmatched trackers: store ghost tracks
                for t_idx in list(unmatched_trks_set):
                    trk = trks[t_idx]
                    if trk.time_since_update <= max_age:
                        pred_box = trk.history[-1]
                        trk.ghost_tracks.append((fid, pred_box))

                # 6. Remove dead trackers
                trackers_by_label[label] = [
                    t for t in trks if t.time_since_update <= max_age
                ]

        print(f"    SORT (max_age={max_age}): "
              f"Removed {duplicates_removed} duplicates, "
              f"rectified {count_fixed} missed detections.")
        return repaired_preds

    # ==================================================================
    # Method 7: SORT + Linear Interpolation (Hybrid)
    # ==================================================================
    def hybrid_sort_interp(self, preds_video, max_age=25,
                           overlap_thresh=0.7):
        """
        Method 7: Hybrid SORT + Linear Interpolation.

        Like Method 6, but instead of using Kalman-predicted ghost
        tracks, it linearly interpolates between the last actual
        detection and the new detection when a tracker is re-matched.

        Also uses stronger NMS with both IoU and IoM (Intersection
        over Minimum area) thresholds.

        Requires: pip install filterpy

        Args:
            preds_video: {frame_id: [detections]} dict
            max_age: max frames a tracker survives without detection (default 25)
            overlap_thresh: IoU threshold for intra-class NMS (default 0.7)
        Returns:
            new preds_video built from tracked + interpolated detections
        """
        if not _SORT_AVAILABLE:
            print("    ⚠️ SORT unavailable (filterpy not installed). Skipping.")
            return preds_video

        sorted_fids = sorted(preds_video.keys())
        no_frames = max(sorted_fids) if sorted_fids else 0

        repaired_preds = {i: [] for i in range(1, no_frames + 1)}
        trackers_by_label = {l: [] for l in range(1, self.num_classes)}
        count_fixed = 0
        duplicates_removed = 0

        for fid in range(1, no_frames + 1):
            current_detections = preds_video.get(fid, [])

            for label in trackers_by_label:
                raw_dets = [
                    d for d in current_detections
                    if int(np.argmax(d['scores'])) == label
                ]

                # Intra-class NMS (IoU + IoM)
                dets = []
                for d in raw_dets:
                    is_dup = False
                    for fd in dets:
                        iou, iom = self._calculate_overlap_metrics(
                            d['bbox'], fd['bbox']
                        )
                        if iou > overlap_thresh or iom > 0.85:
                            is_dup = True
                            duplicates_removed += 1
                            area_d = d['bbox'][2] * d['bbox'][3]
                            area_fd = fd['bbox'][2] * fd['bbox'][3]
                            if area_d > area_fd:
                                fd['bbox'] = d['bbox']
                                fd['bbox_center'] = d.get('bbox_center',
                                    fd.get('bbox_center'))
                            break
                    if not is_dup:
                        dets.append(d)

                trks = trackers_by_label[label]
                predicted_boxes = [trk.predict() for trk in trks]

                # Match using Center Distance + Hungarian
                matched = []
                unmatched_dets_set = set(range(len(dets)))
                unmatched_trks_set = set(range(len(trks)))

                if len(dets) > 0 and len(trks) > 0:
                    cost = np.zeros((len(dets), len(trks)))
                    for d_idx, d in enumerate(dets):
                        for t_idx, t_box in enumerate(predicted_boxes):
                            cost[d_idx, t_idx] = \
                                self._box_center_distance(d['bbox'], t_box)

                    row_ind, col_ind = _linear_sum_assignment(cost)
                    for r, c in zip(row_ind, col_ind):
                        if cost[r, c] < 150.0:
                            matched.append((r, c))
                            unmatched_dets_set.discard(r)
                            unmatched_trks_set.discard(c)

                unmatched_dets = list(unmatched_dets_set)

                # 3. Matched: INTERPOLATE the gap
                for d_idx, t_idx in matched:
                    trk = trks[t_idx]
                    curr_bbox = np.array(dets[d_idx]['bbox'])

                    gap_size = fid - trk.last_actual_fid

                    if 1 < gap_size <= max_age:
                        start_box = trk.last_actual_bbox
                        end_box = curr_bbox
                        start_fid = trk.last_actual_fid

                        for tj in range(start_fid + 1, fid):
                            w1 = (fid - tj) / float(gap_size)
                            w2 = (tj - start_fid) / float(gap_size)
                            bj = (w1 * start_box) + (w2 * end_box)

                            avg_cx = bj[0] + (bj[2] / 2.0)
                            avg_cy = bj[1] + (bj[3] / 2.0)

                            new_scores = np.zeros(self.num_classes,
                                                  dtype=float)
                            new_scores[label] = 1.0
                            new_entry = {
                                'image_id': tj,
                                'bbox': bj.tolist(),
                                'bbox_center': [float(avg_cx),
                                                float(avg_cy)],
                                'scores': new_scores
                            }
                            repaired_preds[tj].append(new_entry)
                            count_fixed += 1

                    trk.update(dets[d_idx]['bbox'], fid)
                    repaired_preds[fid].append(dets[d_idx])

                # 4. Unmatched detections: new trackers
                for d_idx in unmatched_dets:
                    trks.append(
                        _ToolTrackerHybrid(dets[d_idx]['bbox'], label, fid)
                    )
                    repaired_preds[fid].append(dets[d_idx])

                # 5. Remove dead trackers
                trackers_by_label[label] = [
                    t for t in trks if t.time_since_update <= max_age
                ]

        print(f"    Hybrid SORT+Interp (max_age={max_age}): "
              f"Removed {duplicates_removed} duplicates, "
              f"rectified {count_fixed} missed detections.")
        return repaired_preds

In [ ]:
# ================================================================
# Create Processor
# ================================================================

processor = GapFillingProcessor(
    dsu_corrector=dsu_corrector,
    images_root=NEW_IMAGES_ROOT,
    num_classes=6
)

In [ ]:
# ================================================================
# METHOD 1: Gap Fill ±2 neighbors
# ================================================================

processor.run_method(
    "DSU + Gap Fill ±2 + DSU",
    processor.gap_fill_neighbors,
    window=2
)

In [ ]:
# ================================================================
# Visualize Method (Method 1)
# ================================================================

# 1. Choose your surgery
surgery = "surgery_10"

# 2. Extract the predictions for the specific method from the processor
sort_pv = processor.get_preds_video("DSU + Gap Fill ±2 + DSU", surgery)

# 3. Extract your baseline and DSU-only predictions for comparison
baseline_pv = predictor.get_preds_video(surgery)
dsu_pv = dsu_corrector.get_corrected(surgery)


# --- Example: Side-by-side comparison (Raw vs Hybrid SORT) ---
# Find out how many frames there are in total
total_frames = max(baseline_pv.keys()) if baseline_pv else 0

# Compare all the frames
print(f"Comparing {total_frames} frames side-by-side...")
for frame_id in range(1, total_frames + 1):
    viz.compare(
        surgery_name=surgery,
        preds_video_1=dsu_pv,
        preds_video_2=sort_pv,
        frame_id=frame_id,   # Change this to whatever frame you want to inspect
        titles=["After First DSU", "Method 1: Temporal Interp"]
    )


# --- Example: Show a grid of the Hybrid SORT results ---
# viz.show_grid(surgery, sort_pv, frame_ids=[1, 5, 10, 15])


# --- Example: Compare all 3 stages for a single frame! ---
# You can even do this if you want to trace the pipeline's progress:
# viz.show_frame(surgery, baseline_pv, frame_id=15)
# viz.show_frame(surgery, dsu_pv, frame_id=15)
# viz.show_frame(surgery, sort_pv, frame_id=15)


In [ ]:
# ================================================================
# METHOD 2: Advanced Temporal Interpolation
# ================================================================

processor.run_method(
    "DSU + Adv Temporal ±4 + DSU",
    processor.advanced_temporal_interp,
    window=4,
    max_pixel_jump=150.0
)

In [ ]:
# ================================================================
# Visualize Method (Method 2)
# ================================================================

# 1. Choose your surgery
surgery = "surgery_10"

# 2. Extract the predictions for the specific method from the processor
sort_pv = processor.get_preds_video("DSU + Adv Temporal ±4 + DSU", surgery)

# 3. Extract your baseline and DSU-only predictions for comparison
baseline_pv = predictor.get_preds_video(surgery)
dsu_pv = dsu_corrector.get_corrected(surgery)


# --- Example: Side-by-side comparison (Raw vs Hybrid SORT) ---
# Find out how many frames there are in total
total_frames = max(baseline_pv.keys()) if baseline_pv else 0

# Compare all the frames
print(f"Comparing {total_frames} frames side-by-side...")
for frame_id in range(1, total_frames + 1):
    viz.compare(
        surgery_name=surgery,
        preds_video_1=dsu_pv,
        preds_video_2=sort_pv,
        frame_id=frame_id,   # Change this to whatever frame you want to inspect
        titles=["After First DSU", "Method 2: Adv Temporal Interp"]
    )


# --- Example: Show a grid of the Hybrid SORT results ---
# viz.show_grid(surgery, sort_pv, frame_ids=[1, 5, 10, 15])


# --- Example: Compare all 3 stages for a single frame! ---
# You can even do this if you want to trace the pipeline's progress:
# viz.show_frame(surgery, baseline_pv, frame_id=15)
# viz.show_frame(surgery, dsu_pv, frame_id=15)
# viz.show_frame(surgery, sort_pv, frame_id=15)


In [ ]:
# ================================================================
# METHOD 3: Linear Interpolation (Paper Eq. 2)
# ================================================================

processor.run_method(
    "DSU + Linear Interp + DSU",
    processor.linear_interp_paper,
    max_gap=25
)

In [ ]:
# ================================================================
# Visualize Method (Method 3)
# ================================================================

# 1. Choose your surgery
surgery = "surgery_10"

# 2. Extract the predictions for the specific method from the processor
sort_pv = processor.get_preds_video("DSU + Linear Interp + DSU", surgery)

# 3. Extract your baseline and DSU-only predictions for comparison
baseline_pv = predictor.get_preds_video(surgery)
dsu_pv = dsu_corrector.get_corrected(surgery)


# --- Example: Side-by-side comparison (Raw vs Hybrid SORT) ---
# Find out how many frames there are in total
total_frames = max(baseline_pv.keys()) if baseline_pv else 0

# Compare all the frames
print(f"Comparing {total_frames} frames side-by-side...")
for frame_id in range(1, total_frames + 1):
    viz.compare(
        surgery_name=surgery,
        preds_video_1=dsu_pv,
        preds_video_2=sort_pv,
        frame_id=frame_id,   # Change this to whatever frame you want to inspect
        titles=["After First DSU", "Method 3: Linear Interp"]
    )


# --- Example: Show a grid of the Hybrid SORT results ---
# viz.show_grid(surgery, sort_pv, frame_ids=[1, 5, 10, 15])


# --- Example: Compare all 3 stages for a single frame! ---
# You can even do this if you want to trace the pipeline's progress:
# viz.show_frame(surgery, baseline_pv, frame_id=15)
# viz.show_frame(surgery, dsu_pv, frame_id=15)
# viz.show_frame(surgery, sort_pv, frame_id=15)


In [ ]:
# ================================================================
# METHOD 4: Advanced Linear Interpolation (Paper Eq. 2 + Distance)
# ================================================================

processor.run_method(
    "DSU + Adv Linear Interp + DSU",
    processor.advanced_linear_interp,
    max_gap=25,
    max_pixel_jump=150.0
)

In [ ]:
# ================================================================
# Visualize Method (Method 4)
# ================================================================

# 1. Choose your surgery
surgery = "surgery_10"

# 2. Extract the predictions for the specific method from the processor
sort_pv = processor.get_preds_video("DSU + Adv Linear Interp + DSU", surgery)

# 3. Extract your baseline and DSU-only predictions for comparison
baseline_pv = predictor.get_preds_video(surgery)
dsu_pv = dsu_corrector.get_corrected(surgery)


# --- Example: Side-by-side comparison (Raw vs Hybrid SORT) ---
# Find out how many frames there are in total
total_frames = max(baseline_pv.keys()) if baseline_pv else 0

# Compare all the frames
print(f"Comparing {total_frames} frames side-by-side...")
for frame_id in range(1, total_frames + 1):
    viz.compare(
        surgery_name=surgery,
        preds_video_1=dsu_pv,
        preds_video_2=sort_pv,
        frame_id=frame_id,   # Change this to whatever frame you want to inspect
        titles=["After First DSU", "Method 4: Adv Linear Interp"]
    )


# --- Example: Show a grid of the Hybrid SORT results ---
# viz.show_grid(surgery, sort_pv, frame_ids=[1, 5, 10, 15])


# --- Example: Compare all 3 stages for a single frame! ---
# You can even do this if you want to trace the pipeline's progress:
# viz.show_frame(surgery, baseline_pv, frame_id=15)
# viz.show_frame(surgery, dsu_pv, frame_id=15)
# viz.show_frame(surgery, sort_pv, frame_id=15)


In [ ]:
# ================================================================
# METHOD 5: Template Matching (Visual Tracking)
# ================================================================

processor.run_method(
    "DSU + Template Matching + DSU",
    processor.template_matching,
    max_gap=25,
    match_threshold=0.4
)

In [ ]:
# ================================================================
# Visualize Method (Method 5)
# ================================================================

# 1. Choose your surgery
surgery = "surgery_10"

# 2. Extract the predictions for the specific method from the processor
sort_pv = processor.get_preds_video("DSU + Template Matching + DSU", surgery)

# 3. Extract your baseline and DSU-only predictions for comparison
baseline_pv = predictor.get_preds_video(surgery)
dsu_pv = dsu_corrector.get_corrected(surgery)


# --- Example: Side-by-side comparison (Raw vs Hybrid SORT) ---
# Find out how many frames there are in total
total_frames = max(baseline_pv.keys()) if baseline_pv else 0

# Compare all the frames
print(f"Comparing {total_frames} frames side-by-side...")
for frame_id in range(1, total_frames + 1):
    viz.compare(
        surgery_name=surgery,
        preds_video_1=dsu_pv,
        preds_video_2=sort_pv,
        frame_id=frame_id,   # Change this to whatever frame you want to inspect
        titles=["Raw Model", "Hybrid SORT"]
    )


# --- Example: Show a grid of the Hybrid SORT results ---
# viz.show_grid(surgery, sort_pv, frame_ids=[1, 5, 10, 15])


# --- Example: Compare all 3 stages for a single frame! ---
# You can even do this if you want to trace the pipeline's progress:
# viz.show_frame(surgery, baseline_pv, frame_id=15)
# viz.show_frame(surgery, dsu_pv, frame_id=15)
# viz.show_frame(surgery, sort_pv, frame_id=15)


In [ ]:
# ================================================================
# METHOD 6: SORT Tracking (Kalman Filter + Ghost Tracks)
# ================================================================

processor.run_method(
    "DSU + SORT + DSU",
    processor.sort_tracking,
    max_age=25,
    overlap_thresh=0.5
)

In [ ]:
# ================================================================
# Visualize Method (Method 6)
# ================================================================

# 1. Choose your surgery
surgery = "surgery_10"

# 2. Extract the predictions for the specific method from the processor
sort_pv = processor.get_preds_video("DSU + SORT + DSU", surgery)

# 3. Extract your baseline and DSU-only predictions for comparison
baseline_pv = predictor.get_preds_video(surgery)
dsu_pv = dsu_corrector.get_corrected(surgery)


# --- Example: Side-by-side comparison (Raw vs Hybrid SORT) ---
# Find out how many frames there are in total
total_frames = max(baseline_pv.keys()) if baseline_pv else 0

# Compare all the frames
print(f"Comparing {total_frames} frames side-by-side...")
for frame_id in range(1, total_frames + 1):
    viz.compare(
        surgery_name=surgery,
        preds_video_1=dsu_pv,
        preds_video_2=sort_pv,
        frame_id=frame_id,   # Change this to whatever frame you want to inspect
        titles=["After First DSU", "Method 6: SORT Tracking"]
    )


# --- Example: Show a grid of the Hybrid SORT results ---
# viz.show_grid(surgery, sort_pv, frame_ids=[1, 5, 10, 15])


# --- Example: Compare all 3 stages for a single frame! ---
# You can even do this if you want to trace the pipeline's progress:
# viz.show_frame(surgery, baseline_pv, frame_id=15)
# viz.show_frame(surgery, dsu_pv, frame_id=15)
# viz.show_frame(surgery, sort_pv, frame_id=15)


In [ ]:
# ================================================================
# METHOD 7: Hybrid SORT + Linear Interpolation
# ================================================================
processor.run_method(
    "DSU + Hybrid SORT + DSU",
    processor.hybrid_sort_interp,
    max_age=25,
    overlap_thresh=0.7
)

In [ ]:
# ================================================================
# Visualize Method (Method 7)
# ================================================================

# 1. Choose your surgery
surgery = "surgery_10"

# 2. Extract the predictions for the specific method from the processor
sort_pv = processor.get_preds_video("DSU + Hybrid SORT + DSU", surgery)

# 3. Extract your baseline and DSU-only predictions for comparison
baseline_pv = predictor.get_preds_video(surgery)
dsu_pv = dsu_corrector.get_corrected(surgery)


# --- Example: Side-by-side comparison (Raw vs Hybrid SORT) ---
# Find out how many frames there are in total
total_frames = max(baseline_pv.keys()) if baseline_pv else 0

# Compare all the frames
print(f"Comparing {total_frames} frames side-by-side...")
for frame_id in range(1, total_frames + 1):
    viz.compare(
        surgery_name=surgery,
        preds_video_1=baseline_pv,
        preds_video_2=sort_pv,
        frame_id=frame_id,   # Change this to whatever frame you want to inspect
        titles=["Raw Model", "Hybrid SORT"]
    )


# --- Example: Show a grid of the Hybrid SORT results ---
# viz.show_grid(surgery, sort_pv, frame_ids=[1, 5, 10, 15])


# --- Example: Compare all 3 stages for a single frame! ---
# You can even do this if you want to trace the pipeline's progress:
# viz.show_frame(surgery, baseline_pv, frame_id=15)
# viz.show_frame(surgery, dsu_pv, frame_id=15)
# viz.show_frame(surgery, sort_pv, frame_id=15)


In [ ]:
# Save ALL results at once
POSTPROC_OUTPUT_DIR = "/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/postprocessing_results"
processor.save_results(POSTPROC_OUTPUT_DIR)

# List all completed methods
processor.list_methods()

# ================================================================
# VISUALIZE: Compare any method vs DSU checkpoint (optional)
# ================================================================
surgery = gt_extractor.surgery_dirs[0]
gt = gt_extractor.get_surgery_gt(surgery)
dsu_pv = dsu_corrector.get_corrected(surgery)

# Compare Method 1 vs Method 2
# m1_pv = processor.get_preds_video("DSU + Gap Fill ±2 + DSU", surgery)
# m2_pv = processor.get_preds_video("DSU + Adv Temporal ±4 + DSU", surgery)
# viz.compare(surgery, m1_pv, m2_pv, frame_id=1,
#             titles=["Gap Fill ±2", "Adv Temporal ±4"], gt=gt)


# Compare method 1 with DSU checkpoint
# viz.compare(surgery, dsu_pv, m1_pv, frame_id=1,
#             titles=["After DSU", "DSU + Gap Fill ±2 + DSU"], gt=gt)


In [ ]:
# ================================================================
# Tool Usage Analyzer Class
# ================================================================

import json
import os
import numpy as np


class ToolUsageAnalyzer:
    """
    Analyzes surgical tool usage patterns from preds_video predictions.

    Computes:
      - Per-frame binary presence for each tool
      - On/Off intervals (start frame, end frame)
      - Total usage time per tool (in seconds)

    Stores results per (method_name, surgery_name) for cross-method comparison.

    Usage:
        analyzer = ToolUsageAnalyzer(label_to_name={1: 'Suction', ...})

        # Analyze a single preds_video:
        analyzer.analyze("Raw Model", "surgery_1", preds_video, no_frames)

        # Batch analyze from GapFillingProcessor:
        analyzer.analyze_from_processor(processor)

        # Print report:
        analyzer.print_report("Raw Model", "surgery_1")

        # Compare across methods:
        analyzer.print_comparison("surgery_1")

        # Save:
        analyzer.save_reports("/path/to/output")
    """

    def __init__(self, label_to_name, frame_rate=25):
        """
        Args:
            label_to_name: dict mapping label IDs to names
                           e.g. {1: 'Suction', 2: 'Bipolar Forceps', ...}
            frame_rate: video frame rate in fps (default 25)
        """
        self.label_to_name = label_to_name
        self.tool_names = list(label_to_name.values())
        self.frame_rate = frame_rate

        # {method_name: {surgery_name: report_dict}}
        self.all_reports = {}

    # ==================================================================
    # Core: Analyze a single preds_video
    # ==================================================================
    def analyze(self, method_name, surgery_name, preds_video, no_frames=None):
        """
        Analyze tool usage from a preds_video dict.

        Args:
            method_name: identifier (e.g., "Raw Model", "DSU + SORT + DSU")
            surgery_name: surgery directory name
            preds_video: {frame_id: [detections]} dict
            no_frames: total number of frames (auto-detected if None)

        Returns:
            report dict with keys:
              - 'presence': {tool_name: {frame_id: 0 or 1}}
              - 'intervals': {tool_name: [(start, end), ...]}
              - 'total_time_sec': {tool_name: float}
              - 'total_frames': {tool_name: int}
        """
        if no_frames is None:
            sorted_fids = sorted(preds_video.keys())
            no_frames = max(sorted_fids) if sorted_fids else 0

        # --- Step 1: Map presence per frame ---
        presence = {name: {} for name in self.tool_names}

        for frame_id in range(1, no_frames + 1):
            present_set = set()

            if frame_id in preds_video:
                for pred in preds_video[frame_id]:
                    if 'scores' in pred:
                        tool_label = int(np.argmax(pred['scores']))
                        if tool_label in self.label_to_name:
                            present_set.add(
                                self.label_to_name[tool_label]
                            )

            for tool_name in self.tool_names:
                presence[tool_name][frame_id] = (
                    1 if tool_name in present_set else 0
                )

        # --- Step 2: Calculate on/off intervals ---
        intervals = {name: [] for name in self.tool_names}
        total_frames_used = {name: 0 for name in self.tool_names}

        for tool_name in self.tool_names:
            tool_pres = presence[tool_name]
            is_on = False
            start_frame = -1
            duration = 0

            for frame_id in range(1, no_frames + 1):
                present = tool_pres.get(frame_id, 0)

                if present == 1 and not is_on:
                    is_on = True
                    start_frame = frame_id
                elif present == 0 and is_on:
                    is_on = False
                    end_frame = frame_id - 1
                    intervals[tool_name].append((start_frame, end_frame))
                    duration += (end_frame - start_frame + 1)

            # Handle tool still active at last frame
            if is_on:
                end_frame = no_frames
                intervals[tool_name].append((start_frame, end_frame))
                duration += (end_frame - start_frame + 1)

            total_frames_used[tool_name] = duration

        # Build total time in seconds
        total_time_sec = {
            name: frames / float(self.frame_rate)
            for name, frames in total_frames_used.items()
        }

        report = {
            'method_name': method_name,
            'surgery_name': surgery_name,
            'no_frames': no_frames,
            'frame_rate': self.frame_rate,
            'presence': presence,
            'intervals': intervals,
            'total_frames': total_frames_used,
            'total_time_sec': total_time_sec
        }

        # Store
        if method_name not in self.all_reports:
            self.all_reports[method_name] = {}
        self.all_reports[method_name][surgery_name] = report

        return report

    # ==================================================================
    # Batch: Analyze from GapFillingProcessor
    # ==================================================================
    def analyze_from_processor(self, processor):
        """
        Analyze all methods and surgeries from a GapFillingProcessor.

        Args:
            processor: GapFillingProcessor instance (with results populated)
        """
        for method_name, surgery_results in processor.results.items():
            for surgery_name, pv in surgery_results.items():
                self.analyze(method_name, surgery_name, pv)
        print(f"✅ Analyzed {len(processor.results)} methods "
              f"from GapFillingProcessor.")

    # ==================================================================
    # Batch: Analyze from DSULabelCorrector
    # ==================================================================
    def analyze_from_dsu(self, dsu_corrector, method_name="DSU Only"):
        """
        Analyze all surgeries from a DSULabelCorrector checkpoint.

        Args:
            dsu_corrector: DSULabelCorrector instance
            method_name: display name (default "DSU Only")
        """
        for surgery_name in dsu_corrector.corrected_per_surgery:
            pv = dsu_corrector.get_corrected(surgery_name)
            self.analyze(method_name, surgery_name, pv)
        print(f"✅ Analyzed DSU checkpoint "
              f"({len(dsu_corrector.corrected_per_surgery)} surgeries).")

    # ==================================================================
    # Print: Single report
    # ==================================================================
    def print_report(self, method_name, surgery_name):
        """Print a formatted usage report for one method + surgery."""
        if method_name not in self.all_reports:
            print(f"⚠️ Method '{method_name}' not found.")
            return
        if surgery_name not in self.all_reports[method_name]:
            print(f"⚠️ Surgery '{surgery_name}' not found "
                  f"for '{method_name}'.")
            return

        report = self.all_reports[method_name][surgery_name]

        print(f"\n{'=' * 50}")
        print(f"  TOOL USAGE REPORT")
        print(f"  Method:  {method_name}")
        print(f"  Surgery: {surgery_name}")
        print(f"  Frames:  {report['no_frames']} "
              f"@ {report['frame_rate']} fps")
        print(f"{'=' * 50}")

        for tool_name in self.tool_names:
            time_sec = report['total_time_sec'][tool_name]
            n_frames = report['total_frames'][tool_name]
            on_off = report['intervals'][tool_name]

            print(f"\n  [{tool_name.upper()}]")
            print(f"  Total Time: {time_sec:.2f}s "
                  f"({n_frames} frames)")

            if on_off:
                print(f"  Intervals ({len(on_off)}):")
                # Show first 10 intervals, summarize rest
                for i, (start, end) in enumerate(on_off[:10]):
                    dur = (end - start + 1) / self.frame_rate
                    print(f"    > Frame {start:4d} → {end:4d} "
                          f"({dur:.2f}s)")
                if len(on_off) > 10:
                    print(f"    ... and {len(on_off) - 10} more intervals")
            else:
                print("  No usage detected.")

        print(f"\n{'=' * 50}\n")

    # ==================================================================
    # Print: Cross-method comparison table
    # ==================================================================
    def print_comparison(self, surgery_name=None):
        """
        Print a table comparing total usage time across methods.

        Args:
            surgery_name: specific surgery (or None for first available)
        """
        if not self.all_reports:
            print("No reports available.")
            return

        # Determine surgery to compare
        if surgery_name is None:
            first_method = next(iter(self.all_reports))
            surgery_name = next(iter(self.all_reports[first_method]))

        methods = []
        for m in self.all_reports:
            if surgery_name in self.all_reports[m]:
                methods.append(m)

        if not methods:
            print(f"⚠️ No methods found for surgery '{surgery_name}'.")
            return

        # Header
        max_name = max(len(m) for m in methods)
        col_w = max(max(len(n) for n in self.tool_names), 8)

        header = f"{'Method':<{max_name}}  "
        header += "  ".join(f"{n:>{col_w}}" for n in self.tool_names)
        header += f"  {'TOTAL':>{col_w}}"

        print(f"\n{'=' * len(header)}")
        print(f"  Tool Usage Comparison — {surgery_name}")
        print(f"{'=' * len(header)}")
        print(header)
        print("-" * len(header))

        for method in methods:
            report = self.all_reports[method][surgery_name]
            row = f"{method:<{max_name}}  "

            total = 0.0
            for tool_name in self.tool_names:
                t = report['total_time_sec'][tool_name]
                total += t
                row += f"{t:>{col_w}.2f}  "

            row += f"{total:>{col_w}.2f}"
            print(row)

        print(f"{'=' * len(header)}\n")

    # ==================================================================
    # Save reports to JSON
    # ==================================================================
    def save_reports(self, output_dir, method_name=None):
        """
        Save usage reports to JSON files.

        Args:
            output_dir: root output directory
            method_name: save only this method (or all if None)
        """
        os.makedirs(output_dir, exist_ok=True)

        methods = ([method_name] if method_name
                   else list(self.all_reports.keys()))

        for mname in methods:
            if mname not in self.all_reports:
                print(f"⚠️ Method '{mname}' not found, skipping.")
                continue

            for surgery_name, report in self.all_reports[mname].items():
                # Build serializable version (no numpy, no large presence)
                save_data = {
                    'method_name': report['method_name'],
                    'surgery_name': report['surgery_name'],
                    'no_frames': report['no_frames'],
                    'frame_rate': report['frame_rate'],
                    'intervals': {
                        k: [(int(s), int(e)) for s, e in v]
                        for k, v in report['intervals'].items()
                    },
                    'total_frames': {
                        k: int(v)
                        for k, v in report['total_frames'].items()
                    },
                    'total_time_sec': {
                        k: float(v)
                        for k, v in report['total_time_sec'].items()
                    }
                }

                safe_method = (mname.replace(' ', '_').replace('/', '_')
                                   .replace('±', 'pm').replace('+', 'plus'))
                safe_surgery = surgery_name.replace(' ', '_').replace('/', '_')

                filename = f"usage_{safe_method}_{safe_surgery}.json"
                path = os.path.join(output_dir, filename)

                with open(path, 'w') as f:
                    json.dump(save_data, f, indent=2)
                print(f"  📁 Saved: {path}")

        print(f"Usage reports saved to: {output_dir}")

    # ==================================================================
    # Get data for external use
    # ==================================================================
    def get_report(self, method_name, surgery_name):
        """Get report dict for a specific method + surgery."""
        if method_name in self.all_reports:
            return self.all_reports[method_name].get(surgery_name, None)
        return None

    def get_presence_array(self, method_name, surgery_name, tool_name):
        """
        Get binary presence array for a tool.
        Returns: list of 0/1 values, index = frame_id - 1.
        """
        report = self.get_report(method_name, surgery_name)
        if report is None:
            return []
        pres = report['presence'].get(tool_name, {})
        return [pres.get(f, 0) for f in range(1, report['no_frames'] + 1)]

In [ ]:
# ================================================================
# CREATE ANALYZER
# ================================================================

analyzer = ToolUsageAnalyzer(
    label_to_name=LABEL_TO_NAME,
    frame_rate=25
)

# ================================================================
# ANALYZE DSU CHECKPOINT (baseline for comparison)
# ================================================================
analyzer.analyze_from_dsu(dsu_corrector, method_name="DSU Only")

# ================================================================
# ANALYZE ALL GAP-FILLING METHODS
# ================================================================
analyzer.analyze_from_processor(processor)

# ================================================================
# PRINT REPORT FOR A SINGLE METHOD + SURGERY
# ================================================================
surgery = gt_extractor.surgery_dirs[0]
analyzer.print_report("DSU Only", surgery)

# ================================================================
# COMPARE ALL METHODS
# ================================================================
analyzer.print_comparison(surgery)

# ================================================================
# SAVE ALL REPORTS
# ================================================================
USAGE_OUTPUT_DIR = "/content/gdrive/MyDrive/PE_Saumya_Shankar/Resources/usage_reports"
analyzer.save_reports(USAGE_OUTPUT_DIR)

In [ ]:
# ================================================================
# EVALUATE DSU CHECKPOINT (Baseline before gap-filling)
# ================================================================
ground_truth_combined = gt_extractor.get_combined()

dsu_eval_preds = dsu_corrector.get_eval_predictions()
evaluator.evaluate("DSU Only", dsu_eval_preds, ground_truth_combined)

In [ ]:
# ================================================================
# Evaluate all Post-Processing Methods (from GapFillingProcessor)
# ================================================================

ground_truth_combined = gt_extractor.get_combined()

for method_name in processor.results:
    eval_preds = processor.get_eval_predictions(method_name)
    evaluator.evaluate(method_name, eval_preds, ground_truth_combined)


In [ ]:
# ================================================================
# Compare all methods + Export
# ================================================================

evaluator.print_comparison()
evaluator.export_latex()
evaluator.export_per_class_latex("Raw Model")  # Per-class table for baseline
evaluator.save_all_results()                   # Save combined JSON